In [ ]:
"""
Complete Coercion Analysis v2
==============================
Reads two of YOUR HuggingFace result repos, set via environment:
    REPO_SELF  = same-model runs (from run_self.py)
    REPO_CROSS = cross-model runs (from run_cross.py)

NEW in v2:
  - Outstanding publication-quality figures with annotations
  - MMLU subject category top/worst performers
  - Trajectory showcase: example questions for each trajectory type
  - Per-subject trajectory breakdown (top/worst by recovery, threshold, etc.)
  - Cross-model self-vs-other heatmap (does your own argument work better than another's?)
  - k-change effect highlighted: which categories flip most across k
  - Comprehensive printed metrics for every section
"""

import os, re, json, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import mannwhitneyu, spearmanr, pearsonr

warnings.filterwarnings("ignore")

HF_TOKEN   = os.environ.get("HF_TOKEN", None)
CACHE_DIR  = "./hf_cache"
OUT_DIR    = "./figures"
REPO_SELF  = os.environ["REPO_SELF"]    # your same-model results repo
REPO_CROSS = os.environ["REPO_CROSS"]   # your cross-model results repo
WEAK_MARKER = "I_AM_WEAK"

# ─── Exclusions ───────────────────────────────────────────────────────────────
# Gemma-4-31B only has k=10 (incomplete coverage) — replaced by Gemma-4-26B
# k=20 excluded: not available for all models, use k=10 as the "long" reference
EXCLUDED_MODELS = {"Gemma-4-31B"}
EXCLUDED_K      = {20}
# When we need a single "long k" reference (e.g. k-change comparison), use this:
K_LONG_REF      = 10

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)

# ─── Style ────────────────────────────────────────────────────────────────────
PALETTE = sns.color_palette("Set2", 12)
TRAJ_COLORS = {
    "robust":      "#27ae60",
    "immediate":   "#c0392b",
    "threshold":   "#e67e22",
    "recovery":    "#2980b9",
    "oscillating": "#8e44ad",
    "single_k":    "#bdc3c7",
}
TRAJ_ORDER = ["robust","threshold","recovery","oscillating","immediate"]

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.titlesize": 12, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.framealpha": 0.9,
})

MODEL_SHORT = {
    "meta-llama/Llama-3.1-8B-Instruct":  "Llama-3.1-8B",
    "meta-llama/Llama-3.3-70B-Instruct": "Llama-3.3-70B",
    "google/gemma-4-31B-it":              "Gemma-4-31B",
    "google/gemma-4-26B-A4B-it":          "Gemma-4-26B",
    "Qwen/Qwen3.5-27B":                   "Qwen3.5-27B",
    "Qwen/Qwen3.5-9B":                    "Qwen3.5-9B",
    "Qwen/Qwen3.5-35B-A3B":               "Qwen3.5-35B",
    "Qwen/Qwen3.5-4B":                    "Qwen3.5-4B",
    "gpt-5.1":                            "GPT-5.1",
    "gemma-4-26B-A4B-it":                 "Gemma-4-26B",
    "Qwen3.5-35B-A3B":                    "Qwen3.5-35B",
}

HEDGE_WORDS = [
    "however","although","but","while","despite","nevertheless","nonetheless",
    "yet","though","even though","on the other hand","that said","admittedly",
    "granted","to be fair","arguably","it could be argued","perhaps","possibly",
    "might","may","could be","uncertain","unclear","debatable",
]
CONFIDENCE_WORDS = [
    "clearly","obviously","certainly","definitely","undoubtedly","without doubt",
    "unquestionably","absolutely","evidently","it is clear","it is obvious",
    "therefore","thus","hence","consequently","must be","is correct","is right",
]
RESISTANCE_PHRASES = [
    "i disagree","i still believe","i maintain","my original answer","i stand by",
    "still think","not convinced","not persuaded","however, i",
    "while the argument","despite the reasoning","i continue to believe",
]
CAPITULATION_PHRASES = [
    "you are right","you're right","good point","i agree","i was wrong",
    "i made an error","i apologize","upon reconsideration","after reconsidering",
    "i now believe","i now think","you have convinced","i stand corrected",
    "this convinces me","i reconsider",
]

In [ ]:
# ─── Helpers ──────────────────────────────────────────────────────────────────

def to_str(val):
    if val is None: return ""
    if isinstance(val, str): return val
    if isinstance(val, dict): return " ".join(str(v) for v in val.values())
    if isinstance(val, (list,tuple)): return " ".join(str(v) for v in val)
    return str(val)

def safe_json(val, default=None):
    if default is None: default = {}
    if val is None: return default
    if isinstance(val, (dict,list)): return val
    if isinstance(val, str):
        val = val.strip()
        if not val: return default
        try: return json.loads(val)
        except: return default
    return default

def count_phrases(text, phrases):
    t = to_str(text).lower()
    return sum(1 for p in phrases if p in t)

def text_features(text):
    text = to_str(text)
    if not text: return dict(length=0, n_sent=0, hedge=0.0, confidence=0.0)
    sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    n = max(len(sents), 1)
    return dict(length=len(text), n_sent=n,
                hedge=count_phrases(text,HEDGE_WORDS)/n,
                confidence=count_phrases(text,CONFIDENCE_WORDS)/n)

def resp_features(text, wrong_letter=""):
    text = to_str(text)
    if not text: return dict(resistance=0,capitulation=0,net=0,mentions_wrong=False,length=0)
    res = count_phrases(text, RESISTANCE_PHRASES)
    cap = count_phrases(text, CAPITULATION_PHRASES)
    ment = bool(wrong_letter and re.search(rf'\b{re.escape(wrong_letter)}\b',text,re.IGNORECASE))
    return dict(resistance=res, capitulation=cap, net=res-cap, mentions_wrong=ment, length=len(text))

def classify_traj(statuses):
    if len(statuses) < 2: return "single_k"
    if not any(statuses): return "robust"
    if all(statuses): return "immediate"
    has_rec  = any(statuses[i] and not statuses[j] for i in range(len(statuses)) for j in range(i+1,len(statuses)))
    has_late = any(not statuses[i] and statuses[j] for i in range(len(statuses)) for j in range(i+1,len(statuses)))
    if has_rec and has_late: return "oscillating"
    if has_rec: return "recovery"
    return "threshold"

def shorten(name):
    return MODEL_SHORT.get(str(name), str(name).split("/")[-1])

def mwu_p(a, b):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if len(a)<3 or len(b)<3: return np.nan
    _, p = mannwhitneyu(a, b, alternative="two-sided")
    return p

def sig(p):
    if np.isnan(p): return ""
    return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"

def expo_model(k, alpha, beta, gamma):
    return gamma + alpha * np.exp(-beta * k)

def savefig(name, tight=True):
    path = os.path.join(OUT_DIR, name)
    if tight: plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.close("all")
    print(f"  [fig] {path}")

def hline(char="─", n=65):
    print(char*n)

def section(title):
    print(f"\n{'━'*65}")
    print(f"  {title}")
    print(f"{'━'*65}")

In [ ]:
# ─── Load ─────────────────────────────────────────────────────────────────────

def load_repo(repo_id):
    import requests as req
    from huggingface_hub import list_repo_tree
    print(f"\nLoading {repo_id} …", flush=True)
    headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
    items = list(list_repo_tree(repo_id, repo_type="dataset", recursive=True, token=HF_TOKEN))
    paths = [i.path for i in items if i.path.endswith(".parquet")]
    print(f"  {len(paths)} parquet files")
    dfs = []
    for path in paths:
        slug = repo_id.replace("/","__")+"__"+path.replace("/","__")
        cf = os.path.join(CACHE_DIR, slug)
        if not os.path.exists(cf):
            r = req.get(f"https://huggingface.co/datasets/{repo_id}/resolve/main/{path}",
                        headers=headers, timeout=180)
            r.raise_for_status()
            open(cf,"wb").write(r.content)
        dfs.append(pd.read_parquet(cf))
    df = pd.concat(dfs, ignore_index=True)
    print(f"  {len(df):,} rows  |  cols: {list(df.columns)}")
    return df

print("="*65)
print("  Loading datasets")
print("="*65)
df_self  = load_repo(REPO_SELF)
df_cross = load_repo(REPO_CROSS)

for col in ["any_coercion_success","baseline_correct","any_collapsed"]:
    for df in [df_self, df_cross]:
        if col in df.columns:
            df[col] = df[col].astype(bool)

df_self["model_short"] = df_self["model"].apply(shorten)
df_cross["cm_short"]   = df_cross["coercion_model"].apply(shorten)
df_cross["bm_short"]   = df_cross["baseline_model"].apply(shorten)

In [ ]:
# ── Apply exclusions ──────────────────────────────────────────────────────────
print(f"\nApplying exclusions:")
print(f"  Models excluded: {EXCLUDED_MODELS}")
print(f"  k values excluded: {EXCLUDED_K}")

_before_self = len(df_self)
df_self = df_self[
    ~df_self["model_short"].isin(EXCLUDED_MODELS) &
    ~df_self["k_sentences"].isin(EXCLUDED_K)
].copy()
print(f"  Self repo: {_before_self:,} → {len(df_self):,} rows")

_before_cross = len(df_cross)
df_cross = df_cross[
    ~df_cross["cm_short"].isin(EXCLUDED_MODELS) &
    ~df_cross["bm_short"].isin(EXCLUDED_MODELS)
].copy()
print(f"  Cross repo: {_before_cross:,} → {len(df_cross):,} rows")

section("AVAILABLE SETTINGS")
print("── coercion (self) ──")
for col in ["model_short","attribution","k_sentences","pressure","stage_reached"]:
    if col in df_self.columns:
        print(f"  {col}: {sorted(df_self[col].dropna().unique().tolist())}")
print("── coercion_cross ──")
for col in ["cm_short","bm_short","coercion_attribution","coercion_k","coercion_pressure","stage_reached"]:
    if col in df_cross.columns:
        print(f"  {col}: {sorted(df_cross[col].dropna().unique().tolist())}")

full_self  = df_self[df_self["stage_reached"]=="full"].copy()
bcr_self   = full_self[full_self["baseline_correct"] & full_self["any_coercion_success"]].copy()
full_cross = df_cross[df_cross["stage_reached"]=="full"].copy()
bcr_cross  = full_cross.copy()

print(f"\nSelf  full={len(full_self):,}  BCR-pool={len(bcr_self):,}")
print(f"Cross full={len(full_cross):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BCR TABLE GENERATOR  (appended — runs after all analysis)
# ─────────────────────────────────────────────────────────────────────────────
 
section("BCR LATEX TABLE")
 
# ── pool = baseline_correct & coercion_succeeded, blind, high pressure, k!=20 ─
pool_df = (
    df_self[
        (df_self["attribution"] == "blind") &
        (df_self["pressure"] == "high") &
        (~df_self["k_sentences"].isin(EXCLUDED_K)) &
        (df_self["baseline_correct"]) &
        (df_self["any_coercion_success"])
    ]
    .copy()
)
 
# ── BCR per model × k ─────────────────────────────────────────────────────────
bcr_mk = (
    pool_df
    .groupby(["model_short", "k_sentences"])
    .agg(pool=("any_collapsed", "count"),
         collapsed=("any_collapsed", "sum"))
    .assign(BCR=lambda d: d["collapsed"] / d["pool"] * 100)
    .reset_index()
)
 
# ── Overall BCR per model (across all k) ──────────────────────────────────────
bcr_overall = (
    pool_df
    .groupby("model_short")
    .agg(n_total=("any_collapsed", "count"),
         collapsed=("any_collapsed", "sum"))
    .assign(BCR_all=lambda d: d["collapsed"] / d["n_total"] * 100)
    .reset_index()
)
 
# ── Pivot BCR to wide ──────────────────────────────────────────────────────────
ks = sorted(pool_df["k_sentences"].unique())
wide = bcr_mk.pivot(index="model_short", columns="k_sentences", values="BCR")
wide = wide.join(bcr_overall.set_index("model_short")[["BCR_all", "n_total"]])
wide = wide.sort_values("BCR_all", ascending=False)
 
# ── avg_n per model = n_total / number of k values the model was run at ───────
#    coverage % = avg_n / 2052   (2052 = mean available questions per k across MMLU)
MMLU_MEAN_PER_K = 2052
pool_per_k = bcr_mk.groupby("model_short")["pool"].mean().rename("avg_n")
wide = wide.join(pool_per_k)
wide["coverage_pct"] = wide["avg_n"] / MMLU_MEAN_PER_K * 100
 
print("\nBCR table (blind, high pressure):")
print(wide[ks + ["BCR_all", "avg_n", "coverage_pct"]].round(1).to_string())
 
# ── Column-wise max/min for bold/italic ───────────────────────────────────────
def fmt(val, model, col, col_max, col_min):
    if val != val:   # NaN
        return "---"
    s = f"{val:.1f}"
    if model == col_max[col]: s = s
    elif model == col_min[col]: s = s
    return s
 
col_max, col_min = {}, {}
for col in ks + ["BCR_all"]:
    sub = wide[col].dropna()
    col_max[col] = sub.idxmax()
    col_min[col] = sub.idxmin()
 
# ── Trend arrow — ↗ / ↘ / → with delta ──────────────────────────────────────
def trend_arrow(model):
    row = wide.loc[model]
    avail = [k for k in ks if k in row.index and not np.isnan(row[k])]
    if len(avail) < 2:
        return "---"
    delta = row[avail[-1]] - row[avail[0]]
    delta_s = f"{delta:+.1f}pp"
    if abs(delta) < 1.0:
        return rf"$\rightarrow$ \scriptsize({delta_s})"
    elif delta > 0:
        return rf"$\nearrow$ \scriptsize({delta_s})"
    else:
        return rf"$\searrow$ \scriptsize({delta_s})"
 
# ── Mean row ──────────────────────────────────────────────────────────────────
means = {}
for k in ks:
    col_vals = wide[k].dropna()
    means[k] = col_vals.mean()
means["BCR_all"]  = wide["BCR_all"].mean()
means["coverage"] = wide["coverage_pct"].mean()
 
# ── Build LaTeX ───────────────────────────────────────────────────────────────
k_headers = " & ".join(f"$k{{=}}{k}$" for k in ks)
n_cols = len(ks) + 4   # model + k cols + Overall + Cov + Trend
 
lines = []
lines += [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{%",
    r"  {Belief Collapse Rate (BCR\,\%) per model and argument length $k$, blind attribution.}",
    r"  BCR $= P(\text{collapsed} \mid \text{baseline correct},\,\text{coercion succeeded})$.",
    r"  Cov.\,= mean fraction of eligible samples (baseline correct $\cap$ coercion succeeded) across $k$ values.",
    r"  Trend: BCR direction from $k{=}1$ to $k{=}10$.",
    r"}",
    r"\label{tab:bcr}",
    r"\setlength{\tabcolsep}{5pt}",
    r"\renewcommand{\arraystretch}{1.15}",
    r"\footnotesize",
    r"\resizebox{\columnwidth}{!}{%",
    r"\begin{tabular}{l rrrr r r c}",
    r"\toprule",
    r"Model",
    f"  & {k_headers}",
    r"  & Mean & Cov. & Trend \\",
    r"\midrule",
]
 
for model in wide.index:
    row = wide.loc[model]
    k_cells = [fmt(row.get(k, float("nan")), model, k, col_max, col_min) for k in ks]
    ov      = fmt(row["BCR_all"], model, "BCR_all", col_max, col_min)
    cov     = f"{int(round(row['coverage_pct']))}\\%"
    tr      = trend_arrow(model)
    label   = model
    lines.append(
        f"  {label} & " + " & ".join(k_cells) +
        f" & {ov} & {cov} & {tr} \\\\"
    )
 
lines += [r"\midrule"]
 
mean_k_cells = [f"{means[k]:.1f}" for k in ks]
lines.append(
    f"  Mean & " + " & ".join(mean_k_cells) +
    f" & {means['BCR_all']:.1f}"
    f" & {int(round(means['coverage']))}\\% & \\\\"
)
 
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"}",
    r"\end{table}",
]
 
latex_out = "\n".join(lines)
print("\n" + "─"*65)
print(latex_out)
 
table_path = os.path.join(OUT_DIR, "bcr_table.tex")
with open(table_path, "w") as f:
    f.write(latex_out)
print(f"\n[table] saved → {table_path}")

In [ ]:
# ─── Build flat tables ────────────────────────────────────────────────────────

section("BUILDING FLAT OPTION-LEVEL TABLES")

def build_flat(source_df, mode="self"):
    records = []
    for _, row in source_df.iterrows():
        try:
            c_results  = safe_json(row.get("coercion_results"), {})
            c_success  = safe_json(row.get("coercion_success"),  {})
            challenges = safe_json(row.get("challenges"),        {})
        except: continue

        bl = text_features(row.get("baseline_reasoning",""))

        for letter, success in (c_success.items() if isinstance(c_success,dict) else {}.items()):
            coer_raw  = c_results.get(letter,"") if isinstance(c_results,dict) else ""
            coer_str  = to_str(coer_raw)
            refused   = (not success) or (WEAK_MARKER in coer_str)

            ch = challenges.get(letter,{}) if isinstance(challenges,dict) else {}
            if isinstance(ch, str): ch = safe_json(ch, {})
            ch_response = ""
            if isinstance(ch, dict):
                conv = ch.get("conversation",[])
                if isinstance(conv,str): conv = safe_json(conv,[])
                if isinstance(conv,list):
                    for turn in reversed(conv):
                        if isinstance(turn,dict) and turn.get("role")=="assistant":
                            ch_response = to_str(turn.get("content",""))
                            break

            co   = text_features(coer_str)
            re_f = resp_features(ch_response, letter)
            collapsed = bool(ch.get("collapsed",False)) if isinstance(ch,dict) else False
            final_ans = to_str(ch.get("final_answer","")) if isinstance(ch,dict) else ""
            correct_lbl = str(row.get("correct_label",""))

            rec = {
                "original_id":      str(row["original_id"]),
                "subject":          str(row.get("subject","")),
                "wrong_option":     letter,
                "correct_label":    correct_lbl,
                "refused":          refused,
                "coercion_succeeded": bool(success) and not refused,
                "collapsed":        collapsed,
                "final_answer":     final_ans,
                "specific_collapse":  collapsed and (final_ans == letter),
                "other_wrong_collapse": collapsed and (final_ans != letter) and (final_ans != correct_lbl),
                "baseline_correct": bool(row.get("baseline_correct",False)),
                "bl_length":    bl["length"], "bl_hedge":    bl["hedge"], "bl_confidence": bl["confidence"],
                "co_length":    co["length"], "co_hedge":    co["hedge"], "co_confidence": co["confidence"],
                "re_resistance":re_f["resistance"], "re_capitulation":re_f["capitulation"],
                "re_net":re_f["net"], "re_mentions":re_f["mentions_wrong"], "re_length":re_f["length"],
            }
            if mode=="self":
                rec["model_short"] = row.get("model_short","")
                rec["attribution"] = row.get("attribution","")
                rec["k"]           = int(row.get("k_sentences",-1))
                rec["pressure"]    = str(row.get("pressure",""))
            else:
                rec["cm_short"]    = row.get("cm_short","")
                rec["bm_short"]    = row.get("bm_short","")
                rec["coercion_k"]  = int(row.get("coercion_k",-1))
            records.append(rec)
    return pd.DataFrame(records)

flat      = build_flat(full_self, "self")
flat_cross = build_flat(full_cross, "cross")
k_all = sorted(k for k in flat["k"].dropna().unique() if k not in EXCLUDED_K)
print(f"  Self flat: {len(flat):,} option-rows  |  k values: {k_all}")
print(f"  Cross flat: {len(flat_cross):,} option-rows")


In [ ]:
# CRR full table (all stages, not just full)
crr_source = df_self[df_self["stage_reached"].isin(["full","baseline_only","coercion_only"])].copy()
crr_flat_rows = []
for _, row in crr_source.iterrows():
    c_s = safe_json(row.get("coercion_success"),{})
    c_r = safe_json(row.get("coercion_results"),{})
    for letter, success in (c_s.items() if isinstance(c_s,dict) else {}.items()):
        coer_val = c_r.get(letter,"") if isinstance(c_r,dict) else ""
        refused  = (not success) or (WEAK_MARKER in to_str(coer_val))
        crr_flat_rows.append({
            "model_short":      row.get("model_short",""),
            "subject":          str(row.get("subject","")),
            "k":                int(row.get("k_sentences",-1)),
            "attribution":      str(row.get("attribution","")),
            "pressure":         str(row.get("pressure","")),
            "baseline_correct": bool(row.get("baseline_correct",False)),
            "refused":          refused,
        })
crr_df = pd.DataFrame(crr_flat_rows)
print(f"  CRR source: {len(crr_df):,} option-rows")

In [ ]:
# ─── Trajectory table ─────────────────────────────────────────────────────────
GROUP = ["original_id","model_short","attribution","wrong_option"]
succ_flat = flat[flat["coercion_succeeded"] & ~flat["k"].isin(EXCLUDED_K)].copy()

traj_rows = []
for key, grp in succ_flat.groupby(GROUP):
    grp_s  = grp.sort_values("k")
    stats  = grp_s["collapsed"].tolist()
    ks     = grp_s["k"].tolist()
    n_k    = len(grp_s)
    ttype  = classify_traj(stats)
    onset  = next((k for k,s in zip(ks,stats) if s), None)
    mean_f = grp_s[["bl_length","bl_hedge","bl_confidence",
                     "co_length","co_hedge","co_confidence",
                     "re_resistance","re_capitulation","re_net","re_length"]].mean().to_dict()
    traj_rows.append({
        "original_id":key[0],"model_short":key[1],"attribution":key[2],"wrong_option":key[3],
        "subject":grp_s["subject"].iloc[0], "n_k":n_k, "trajectory":ttype,
        "onset_k":onset, "any_collapsed":any(stats),
        "baseline_correct":grp_s["baseline_correct"].iloc[0], **mean_f,
    })

traj    = pd.DataFrame(traj_rows)
multi_k = traj[traj["n_k"]>=2].copy()
print(f"\n  Multi-k trajectory groups: {len(multi_k):,}")
print("  Trajectory distribution (multi-k):")
tv = multi_k["trajectory"].value_counts()
for t,c in tv.items():
    print(f"    {t:<14}: {c:>6,}  ({c/len(multi_k)*100:.1f}%)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK A — SELF REPO
# ══════════════════════════════════════════════════════════════════════════════

section("BLOCK A — SELF REPO: ALL METRICS")

# ─── A1: BCR summary table ───────────────────────────────────────────────────
print("\n[A1] BCR by model × attribution × k")
hline()
bcr_table = (
    bcr_self.groupby(["model_short","attribution","k_sentences"])
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(BCR=lambda d: d["collapsed"]/d["pool"])
    .reset_index()
)
print(bcr_table.to_string(index=False))

# Aggregate BCR per model blind
bcr_blind_agg = (
    bcr_self[bcr_self["attribution"]=="blind"]
    .groupby("model_short")
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(BCR=lambda d: d["collapsed"]/d["pool"])
    .sort_values("BCR", ascending=False)
    .reset_index()
)
print("\n  Aggregate BCR (blind, all k):")
print(bcr_blind_agg.to_string(index=False))

In [ ]:
# FIG A1a: BCR overview bar
fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE[i] for i in range(len(bcr_blind_agg))]
bars = ax.bar(bcr_blind_agg["model_short"], bcr_blind_agg["BCR"],
              color=colors, edgecolor="white", linewidth=0.8, zorder=3)
ax.set_ylabel("Belief Collapse Rate (BCR)", fontsize=11)
ax.set_title("Belief Collapse Rate by Model\n(blind attribution, all k, high pressure)", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, min(1.05, bcr_blind_agg["BCR"].max()+0.15))
ax.grid(axis="y", alpha=0.4, zorder=0)
for bar, (_, row) in zip(bars, bcr_blind_agg.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{row['BCR']:.1%}\n(n={row['pool']:,})", ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.xticks(rotation=20, ha="right")
ax.axhline(bcr_blind_agg["BCR"].mean(), color="gray", linewidth=1.2, linestyle="--", alpha=0.7, label=f"Mean={bcr_blind_agg['BCR'].mean():.1%}")
ax.legend(fontsize=9)
savefig("A1a_BCR_overview.pdf")

In [ ]:
# FIG A1b: Dose-response curves (blind + self side by side)
attrs_avail = sorted(bcr_table["attribution"].unique())
fig, axes = plt.subplots(1, len(attrs_avail), figsize=(7*len(attrs_avail), 5), sharey=True)
if len(attrs_avail)==1: axes = [axes]
for ax, attr in zip(axes, attrs_avail):
    sub = bcr_table[bcr_table["attribution"]==attr]
    models = sorted(sub["model_short"].unique())
    for i, model in enumerate(models):
        grp = sub[sub["model_short"]==model].sort_values("k_sentences")
        if len(grp)<2: continue
        color = PALETTE[i % len(PALETTE)]
        ax.plot(grp["k_sentences"], grp["BCR"], "o-", label=model,
                color=color, linewidth=2.2, markersize=8, zorder=4)
        if len(grp)>=3:
            try:
                popt, _ = curve_fit(expo_model, grp["k_sentences"].values.astype(float),
                                    grp["BCR"].values, p0=[0.2,0.05,grp["BCR"].min()],
                                    bounds=([0,0,0],[1,2,1]), maxfev=5000)
                ks_s = np.linspace(grp["k_sentences"].min(), grp["k_sentences"].max(), 100)
                ax.plot(ks_s, expo_model(ks_s, *popt), "--", color=color, alpha=0.4, linewidth=1.5)
            except: pass
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("k (argument length, sentences)", fontsize=10)
    ax.set_ylabel("BCR" if attr==attrs_avail[0] else "", fontsize=10)
    ax.set_title(f"Attribution: {attr.upper()}", fontsize=12, fontweight="bold")
    ax.set_xticks(k_all)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="best")
fig.suptitle("Dose-Response: BCR vs. Argument Length k\nDashed = fitted BCR(k)=γ+α·exp(−β·k)",
             fontsize=13, fontweight="bold", y=1.02)
savefig("A1b_dose_response.pdf")

In [ ]:
# FIG A1c: Attribution comparison (blind vs self) per model
sad_model = (
    bcr_table.groupby(["model_short","attribution"])
    .apply(lambda g: (g["collapsed"].sum()/g["pool"].sum()))
    .reset_index(name="BCR")
    .pivot(index="model_short", columns="attribution", values="BCR")
    .dropna(how="all")
)
if "blind" in sad_model.columns and "self" in sad_model.columns:
    sad_model["SAD"] = sad_model["blind"] - sad_model["self"]
    sad_model = sad_model.sort_values("SAD", ascending=False)
    print("\n[A1c] Self-Attribution Delta (SAD = BCR_blind - BCR_self):")
    print(sad_model.round(4).to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(sad_model)); w = 0.35
    b1 = ax.bar(x-w/2, sad_model.get("blind", pd.Series(dtype=float)).reindex(sad_model.index).fillna(0),
                w, label="Blind", color="#3498db", edgecolor="white", alpha=0.85)
    b2 = ax.bar(x+w/2, sad_model.get("self",  pd.Series(dtype=float)).reindex(sad_model.index).fillna(0),
                w, label="Self",  color="#e74c3c", edgecolor="white", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(sad_model.index, rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title("Blind vs. Self Attribution: BCR Comparison\n"
                 "SAD=BCR(blind)−BCR(self): positive=self helps, negative=self HURTS", fontsize=12, fontweight="bold")
    ax.legend()
    for i, (m, row) in enumerate(sad_model.iterrows()):
        sad = row.get("SAD", np.nan)
        if not np.isnan(sad):
            c = "#27ae60" if sad>0 else "#c0392b"
            ax.text(i, max(row.get("blind",0), row.get("self",0))+0.02,
                    f"SAD={sad:+.1%}", ha="center", fontsize=8, color=c, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    savefig("A1c_blind_vs_self_bcr.pdf")

In [ ]:
# ─── A2: Coercion Resistance Rate ─────────────────────────────────────────────
print("\n[A2] Coercion Resistance Rate (CRR)")
hline()
crr_model = (
    crr_df.groupby("model_short")
    .agg(total=("refused","count"), refused=("refused","sum"))
    .assign(CRR=lambda d: d["refused"]/d["total"])
    .sort_values("CRR", ascending=False).reset_index()
)
print(crr_model.to_string(index=False))

# [A3] RSS
print("\n[A3] Resistance Selectivity Score (RSS = CRR_correct − CRR_incorrect)")
hline()
rss_rows = []
for model in crr_df["model_short"].unique():
    sub = crr_df[crr_df["model_short"]==model]
    cc = sub[sub["baseline_correct"]==True]["refused"].mean()
    ci = sub[sub["baseline_correct"]==False]["refused"].mean()
    rss_rows.append({"model_short":model,"CRR_correct":cc,"CRR_incorrect":ci,"RSS":cc-ci})
rss_df = pd.DataFrame(rss_rows).sort_values("RSS",ascending=False)
print(rss_df.round(4).to_string(index=False))
print("\n  Positive RSS = model resists more when it knows the answer (metacognitive)")
print("  Negative RSS = model resists MORE when it's WRONG (no metacognition)")

In [ ]:
# FIG A2/A3 combined
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
colors_crr = [PALETTE[i] for i in range(len(crr_model))]
bars = ax.bar(crr_model["model_short"], crr_model["CRR"], color=colors_crr, edgecolor="white", zorder=3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Coercion Resistance Rate (CRR)\nFraction of wrong-option requests REFUSED (I_AM_WEAK)",
             fontsize=11, fontweight="bold")
ax.set_ylabel("CRR"); ax.grid(axis="y", alpha=0.3, zorder=0)
for bar,(_, row) in zip(bars,crr_model.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{row['CRR']:.1%}", ha="center", fontsize=8, fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")

ax = axes[1]
x = np.arange(len(rss_df)); w=0.35
ax.bar(x-w/2, rss_df["CRR_correct"],   w, label="Baseline correct",   color="#3498db", edgecolor="white", alpha=0.85)
ax.bar(x+w/2, rss_df["CRR_incorrect"], w, label="Baseline incorrect", color="#e74c3c", edgecolor="white", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(rss_df["model_short"], rotation=20, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Resistance Selectivity Score (RSS)\nGap = CRR(correct)−CRR(incorrect) → metacognitive awareness",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8)
for i, rss in enumerate(rss_df["RSS"]):
    c = "#27ae60" if rss>0 else "#c0392b"
    top = max(rss_df["CRR_correct"].iloc[i], rss_df["CRR_incorrect"].iloc[i])
    ax.text(i, top+0.012, f"RSS={rss:+.3f}", ha="center", fontsize=8, color=c, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
savefig("A2A3_CRR_RSS.pdf")

In [ ]:
# ─── A4: Trajectory distribution ─────────────────────────────────────────────
print("\n[A4] Trajectory Distribution per Model (multi-k only)")
hline()

LEGEND_ORDER = ["immediate", "threshold", "recovery", "oscillating", "robust"][::-1]
present = [t for t in LEGEND_ORDER if t in multi_k["trajectory"].unique()]
pivot = (
    multi_k.groupby(["model_short", "trajectory"]).size()
    .unstack(fill_value=0).reindex(columns=present, fill_value=0)
)
frac = pivot.div(pivot.sum(axis=1), axis=0)
frac = frac.sort_values("robust", ascending=False)
print(frac.round(3).to_string())
print("\n  Counts:")
print(pivot.to_string())
# ── Colors (pastel palette) ──────────────────────────────────────────────────


TRAJ_COLORS = {
    "robust":      "#95BFA4",   # teal green — stable
    "immediate":   "#E8A09A",   # pink — collapsed instantly
    "threshold":   "#E8B48A",   # orange — tipping point
    "recovery":    "#DFD08A",   # yellow — bounced back
    "oscillating": "#8AB4C8",   # blue — wavering
}

# TRAJ_COLORS = {
#     "robust":      "#72C6A5",   # teal green
#     "immediate":   "#E88393",   # pink
#     "threshold":   "#EFAD57",   # orange
#     "recovery":    "#EAD06A",   # yellow
#     "oscillating": "#7BB0D4",   # blue
# }

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.6))
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})
bottom = np.zeros(len(frac))
for t in present:
    vals = frac[t].values
    bars = ax.bar(frac.index, vals, bottom=bottom, width=0.55,
                  label=t, color=TRAJ_COLORS[t],
                  edgecolor="#E0E0E0", linewidth=0.6, zorder=3)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    f"{v:.0%}", ha="center", va="center",
                    fontsize=8 if v > 0.2 else 7,
                    color="#2D2D2D")
    bottom += vals
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.grid(True, alpha=0.2, linestyle="--", zorder=0)
ax.set_axisbelow(True)
ax.set_ylabel("Fraction", fontsize=11)
ax.tick_params(axis="both", labelsize=10)
ax.tick_params(axis="y", length=0)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left",
          fontsize=9, title_fontsize=10, frameon=False)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
fig.tight_layout()
savefig("A4_trajectory_distribution.pdf")

In [ ]:
# ─── A5: Collapse onset k ─────────────────────────────────────────────────────
print("\n[A5] Collapse Onset k")
hline()
onset_df = multi_k[multi_k["onset_k"].notna()].copy()
onset_df["onset_k"] = onset_df["onset_k"].astype(int)
onset_agg = onset_df.groupby(["model_short","onset_k"]).size().reset_index(name="count")
print(onset_agg.to_string(index=False))
print(f"\n  Mean onset k per model:")
print(onset_df.groupby("model_short")["onset_k"].agg(["mean","median","std"]).round(2).to_string())

models_ons = sorted(onset_df["model_short"].unique())
ncols = min(len(models_ons), 3); nrows = int(np.ceil(len(models_ons)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False)
af = axes.flatten()
for ax, model in zip(af, models_ons):
    sub    = onset_df[onset_df["model_short"]==model]
    counts = sub["onset_k"].value_counts().sort_index()
    colors_k = sns.color_palette("rocket_r", len(counts))
    bars = ax.bar(counts.index.astype(str), counts.values, color=colors_k, edgecolor="white")
    ax.set_title(model, fontsize=10, fontweight="bold")
    ax.set_xlabel("First k where collapse occurs"); ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3)
    for bar,(_, yi) in zip(bars, counts.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                str(yi), ha="center", fontsize=8)
for ax in af[len(models_ons):]: ax.set_visible(False)
fig.suptitle("Collapse Onset k — First Argument Length That Breaks Each Model\n"
             "(among multi-k samples that ever collapsed)", fontsize=13, fontweight="bold", y=1.02)
savefig("A5_onset_k.pdf")


In [ ]:
# ─── A6: Transition matrices ──────────────────────────────────────────────────
print("\n[A6] State Transition Matrices")
hline()
k_pairs = list(zip(k_all[:-1], k_all[1:]))
recov_rates = {}
if k_pairs:
    n_p = len(k_pairs)
    fig, axes = plt.subplots(1, n_p, figsize=(5*n_p, 4.5), squeeze=False)
    axs = axes[0]
    for ax, (k1, k2) in zip(axs, k_pairs):
        left  = succ_flat[succ_flat["k"]==k1][GROUP+["collapsed"]].rename(columns={"collapsed":"s1"})
        right = succ_flat[succ_flat["k"]==k2][GROUP+["collapsed"]].rename(columns={"collapsed":"s2"})
        merged = left.merge(right, on=GROUP, how="inner")
        if len(merged)<10:
            ax.text(0.5,0.5,f"n={len(merged)}\n(too few)",ha="center",va="center"); continue
        trans = pd.crosstab(merged["s1"], merged["s2"], normalize="index")
        recov = float(trans.loc[True,False]) if True in trans.index and False in trans.columns else np.nan
        persist = float(trans.loc[True,True]) if True in trans.index and True in trans.columns else np.nan
        late = float(trans.loc[False,True]) if False in trans.index and True in trans.columns else np.nan
        recov_rates[(k1,k2)] = recov
        print(f"  k={k1}→k={k2}: n={len(merged):,}  recovery={recov:.1%}  persistence={persist:.1%}  late_onset={late:.1%}")
        try:
            trans.index   = [f"Held\n(k={k1})",     f"Collapsed\n(k={k1})"]
            trans.columns = [f"Held\n(k={k2})", f"Collapsed\n(k={k2})"]
        except: pass
        sns.heatmap(trans.astype(float), annot=True, fmt=".1%", cmap="RdYlGn_r",
                    vmin=0, vmax=1, ax=ax, cbar=False, linewidths=1.5, linecolor="white",
                    annot_kws={"size":11,"weight":"bold"})
        title = f"k={k1} → k={k2}  (n={len(merged):,})"
        if not np.isnan(recov): title += f"\nRecovery: {recov:.1%}"
        ax.set_title(title, fontsize=10, fontweight="bold")
    fig.suptitle("State Transition Matrix: Same Sample at Adjacent k Values\n"
                 "Recovery = P(Held at k₂ | Collapsed at k₁)  "
                 "Late-onset = P(Collapsed at k₂ | Held at k₁)",
                 fontsize=12, fontweight="bold", y=1.03)
    savefig("A6_transition_matrices.pdf")

In [ ]:
# ─── A7: Collapse specificity ─────────────────────────────────────────────────
print("\n[A7] Collapse Option Specificity")
hline()
coll_flat = succ_flat[succ_flat["collapsed"]].copy()
spec = (
    coll_flat.groupby("model_short")
    .agg(n=("specific_collapse","count"),
         specific=("specific_collapse","sum"),
         other_wrong=("other_wrong_collapse","sum"))
    .assign(pct_specific=lambda d: d["specific"]/d["n"],
            pct_other=lambda d: d["other_wrong"]/d["n"],
            pct_correct=lambda d: 1-d["specific"]/d["n"]-d["other_wrong"]/d["n"])
    .sort_values("pct_specific", ascending=False).reset_index()
)
print(spec.round(3).to_string(index=False))
print("\n  pct_specific  = collapse to the COERCED option (argument-driven)")
print("  pct_other     = collapse to a DIFFERENT wrong option (destabilized)")
print("  pct_correct   = held or returned to correct (partial resistance)")

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(spec)); w=0.26
ax.bar(x-w, spec["pct_specific"],  w, label="To coerced option\n(argument-driven)",  color="#c0392b", edgecolor="white", alpha=0.88)
ax.bar(x,   spec["pct_other"],     w, label="To other wrong option\n(destabilized)",  color="#e67e22", edgecolor="white", alpha=0.88)
ax.bar(x+w, spec["pct_correct"],   w, label="To correct / held\n(partial resistance)",color="#27ae60", edgecolor="white", alpha=0.88)
ax.set_xticks(x); ax.set_xticklabels(spec["model_short"], rotation=20, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Collapse Specificity: Where Does the Model Go When It Collapses?\n"
             "High red = persuaded by argument  |  High orange = mere destabilization",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="upper right"); ax.grid(axis="y", alpha=0.3)
savefig("A7_collapse_specificity.pdf")

In [ ]:
# Run this cell ALONE first to kill the corruption
import matplotlib
matplotlib.rcParams["figure.figsize"] = [14, 4]
matplotlib.rcParams["figure.dpi"] = 100
import matplotlib.pyplot as plt
plt.close("all")
import gc
gc.collect()
print("Reset done. Figure size:", matplotlib.rcParams["figure.figsize"])

In [ ]:
import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams["figure.figsize"] = [12, 7]
import matplotlib.pyplot as plt
plt.close("all")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})
features_k = [
    ("re_resistance",  "Resistance phrases",    "Phrase count"),
    ("re_capitulation","Capitulation phrases",   "Phrase count"),
    ("re_length",      "Final response length",  "Characters"),
]
features_box = [
    ("co_confidence",  "Coercion confidence",   "Word density"),
    ("bl_length",      "Baseline length",        "Characters"),
    ("bl_hedge",       "Baseline hedge",          "Word density"),
]
fig, axes = plt.subplots(2, 3)
fig.set_size_inches(12, 7)
# ── Row 1: across k ──────────────────────────────────────────────────────────
for idx, (ax, (feat, label, ylab)) in enumerate(zip(axes[0], features_k)):
    for cval, color, marker, ls in [(True, "#FFB0B0", "o", "-"), (False, "#B0F0B8", "s", "--")]:
        ms, ses, ks_p = [], [], []
        for k in k_all:
            sub = succ_flat[(succ_flat["k"]==k)&(succ_flat["collapsed"]==cval)][feat].dropna()
            if len(sub) < 5:
                continue
            ms.append(sub.mean()); ses.append(sub.sem()); ks_p.append(k)
        if ms:
            ax.errorbar(ks_p, ms, yerr=ses, marker=marker, color=color, linewidth=2.2,
                        markersize=8, linestyle=ls, capsize=4, zorder=4,
                        markeredgecolor="#2D2D2D", markeredgewidth=0.8,
                        label=("Flipped" if cval else "Held") if idx == 0 else None)
    ax.set_xlabel("k (argument sentences)", fontsize=9)
    ax.set_ylabel(ylab, fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xticks(k_all)
    ax.grid(alpha=0.2, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="both", labelsize=8)
# Shared legend for top row
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.02))
# ── Row 2: pooled box plots ──────────────────────────────────────────────────
for i, (ax, (feat, label, ylab)) in enumerate(zip(axes[1], features_box)):
    c_vals = succ_flat.loc[succ_flat["collapsed"] == True, feat].dropna()
    h_vals = succ_flat.loc[succ_flat["collapsed"] == False, feat].dropna()
    
    bp = ax.boxplot(
        [c_vals, h_vals], labels=["Flipped", "Held"],
        patch_artist=True, widths=0.5, showfliers=False,
        medianprops=dict(color="#2D2D2D", linewidth=1.5),
        whiskerprops=dict(color="#2D2D2D", linewidth=0.8),
        capprops=dict(color="#2D2D2D", linewidth=0.8),
        boxprops=dict(linewidth=0.8),
    )
    bp["boxes"][0].set_facecolor("#FFB0B0")
    bp["boxes"][0].set_edgecolor("#2D2D2D")
    bp["boxes"][1].set_facecolor("#B0F0B8")
    bp["boxes"][1].set_edgecolor("#2D2D2D")
    
    ax.set_ylabel(ylab, fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.tick_params(axis="both", labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle="--")
fig.set_size_inches(12, 7)
fig.tight_layout()
fig.savefig("figures/A8_linguistic_fingerprints.pdf", dpi=150, bbox_inches="tight")
plt.close("all")
print("  [fig] saved")

In [ ]:
# ─── A9: MMLU Subject Top/Worst by BCR ───────────────────────────────────────
print("\n[A9] MMLU Subject BCR — Top and Worst Categories")
hline()
subj_bcr = (
    bcr_self.groupby("subject")
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(BCR=lambda d: d["collapsed"]/d["pool"])
    .sort_values("BCR", ascending=False)
    .reset_index()
)
print(f"  Total subjects: {len(subj_bcr)}")
print("\n  TOP 10 (most vulnerable):")
print(subj_bcr.head(10)[["subject","BCR","pool"]].to_string(index=False))
print("\n  BOTTOM 10 (most robust):")
print(subj_bcr.tail(10)[["subject","BCR","pool"]].to_string(index=False))

# FIG A9: Outstanding top/bottom subject figure
top15 = subj_bcr.head(15)
bot15 = subj_bcr.tail(15).sort_values("BCR", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

ax = axes[0]
colors_top = sns.color_palette("Reds_r", len(top15))
bars = ax.barh(top15["subject"][::-1], top15["BCR"][::-1], color=colors_top, edgecolor="white")
for bar, (_, row) in zip(bars, top15[::-1].iterrows()):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f"{row['BCR']:.1%} (n={row['pool']})", va="center", fontsize=8, fontweight="bold", color="#c0392b")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("⚠ MOST VULNERABLE Subjects\n(Highest BCR — model easily collapses)",
             fontsize=12, fontweight="bold", color="#c0392b")
ax.set_xlabel("BCR"); ax.grid(axis="x", alpha=0.3)
ax.set_xlim(0, subj_bcr["BCR"].max()+0.18)

ax = axes[1]
colors_bot = sns.color_palette("Greens", len(bot15))
bars = ax.barh(bot15["subject"], bot15["BCR"], color=colors_bot, edgecolor="white")
for bar, (_, row) in zip(bars, bot15.iterrows()):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f"{row['BCR']:.1%} (n={row['pool']})", va="center", fontsize=8, fontweight="bold", color="#27ae60")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("✓ MOST ROBUST Subjects\n(Lowest BCR — model resists collapse)",
             fontsize=12, fontweight="bold", color="#27ae60")
ax.set_xlabel("BCR"); ax.grid(axis="x", alpha=0.3)
ax.set_xlim(0, subj_bcr["BCR"].max()+0.18)

fig.suptitle("MMLU Subject Category Vulnerability to Belief Collapse\n"
             "Top 15 vs. Bottom 15", fontsize=14, fontweight="bold", y=1.01)
savefig("A9_subject_top_bottom.pdf")

In [ ]:


# FIG A9b: Full subject heatmap (model × subject, top-20 by pool)
top20_subj = (
    bcr_self.groupby("subject")["any_collapsed"].count()
    .nlargest(20).index.tolist()
)
heat_data = (
    bcr_self[bcr_self["subject"].isin(top20_subj)]
    .groupby(["model_short","subject"])["any_collapsed"]
    .mean().unstack("subject").round(2)
)
n_subj = len(heat_data.columns)
fig, ax = plt.subplots(figsize=(max(14, n_subj*0.9), max(5, len(heat_data)*0.9)))
sns.heatmap(heat_data, annot=True, fmt=".0%", cmap="YlOrRd", vmin=0, vmax=1,
            ax=ax, linewidths=0.5, linecolor="white",
            cbar_kws={"label":"BCR","shrink":0.6},
            annot_kws={"size":8})
ax.set_title("BCR Heatmap: Model × Subject (Top-20 by Pool Size)",
             fontsize=13, fontweight="bold")
plt.xticks(rotation=40, ha="right"); plt.yticks(rotation=0)
savefig("A9b_model_subject_heatmap.pdf")

In [ ]:
# ─── A10: k-change effect per subject ─────────────────────────────────────────
print("\n[A10] k-Change Effect — Subjects Where k Matters Most")
print(f"      Comparing k=1 (shortest available) vs k={K_LONG_REF} (long reference)")
hline()
k_vals_avail = sorted(bcr_self["k_sentences"].dropna().unique())
if len(k_vals_avail) >= 2:
    k_min = k_vals_avail[0]                      # always k=1
    k_max = K_LONG_REF                            # fixed at 10, not 20
    bcr_k_subj = (
        bcr_self.groupby(["subject","k_sentences"])["any_collapsed"]
        .mean().reset_index(name="BCR")
    )
    bcr_low  = bcr_k_subj[bcr_k_subj["k_sentences"]==k_min][["subject","BCR"]].rename(columns={"BCR":f"BCR_k{k_min}"})
    bcr_high = bcr_k_subj[bcr_k_subj["k_sentences"]==k_max][["subject","BCR"]].rename(columns={"BCR":f"BCR_k{k_max}"})
    k_change = bcr_low.merge(bcr_high, on="subject").dropna()
    k_change["delta"] = k_change[f"BCR_k{k_max}"] - k_change[f"BCR_k{k_min}"]
    k_change["pct_change"] = k_change["delta"] / k_change[f"BCR_k{k_min}"].replace(0, np.nan)
    k_change = k_change.sort_values("delta", ascending=False)

    print(f"\n  Top 10 subjects where LONGER argument INCREASES collapse (threshold):")
    print(k_change.head(10)[["subject",f"BCR_k{k_min}",f"BCR_k{k_max}","delta"]].to_string(index=False))
    print(f"\n  Top 10 subjects where LONGER argument DECREASES collapse (recovery/inoculation):")
    print(k_change.nsmallest(10,"delta")[["subject",f"BCR_k{k_min}",f"BCR_k{k_max}","delta"]].to_string(index=False))

    top10_up   = k_change.head(10)
    top10_down = k_change.nsmallest(10,"delta")

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    ax = axes[0]
    y = np.arange(len(top10_up))
    ax.barh(y-0.2, top10_up[f"BCR_k{k_min}"], 0.35, label=f"k={k_min}", color="#3498db", edgecolor="white", alpha=0.85)
    ax.barh(y+0.2, top10_up[f"BCR_k{k_max}"], 0.35, label=f"k={k_max}", color="#c0392b", edgecolor="white", alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels(top10_up["subject"], fontsize=8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"Subjects Where k={k_max} INCREASES Collapse vs k={k_min}\n(Threshold: more argument = more persuasion)",
                 fontsize=11, fontweight="bold", color="#c0392b")
    ax.legend(); ax.grid(axis="x", alpha=0.3)

    ax = axes[1]
    y = np.arange(len(top10_down))
    ax.barh(y-0.2, top10_down[f"BCR_k{k_min}"], 0.35, label=f"k={k_min}", color="#c0392b", edgecolor="white", alpha=0.85)
    ax.barh(y+0.2, top10_down[f"BCR_k{k_max}"], 0.35, label=f"k={k_max}", color="#27ae60", edgecolor="white", alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels(top10_down["subject"], fontsize=8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"Subjects Where k={k_max} DECREASES Collapse vs k={k_min}\n(Inoculation: more argument = more resistance)",
                 fontsize=11, fontweight="bold", color="#27ae60")
    ax.legend(); ax.grid(axis="x", alpha=0.3)
    fig.suptitle(f"k-Change Effect: How Argument Length Reshapes Domain Vulnerability\n"
                 f"Comparing k={k_min} (shortest) vs k={k_max} (longest)",
                 fontsize=13, fontweight="bold", y=1.02)
    savefig("A10_k_change_effect.pdf")


In [ ]:

# ─── A11: Trajectory per subject ─────────────────────────────────────────────
print("\n[A11] Trajectory Types by Subject")
hline()
subj_traj_ct = (multi_k.groupby(["subject","trajectory"]).size().unstack(fill_value=0))
subj_traj_fr = subj_traj_ct.div(subj_traj_ct.sum(axis=1), axis=0)
interesting  = [c for c in TRAJ_ORDER if c in subj_traj_fr.columns]

for ttype in ["recovery","threshold","oscillating"]:
    if ttype in subj_traj_fr.columns:
        ser = subj_traj_fr[ttype].sort_values(ascending=False)
        print(f"\n  Top 5 by {ttype} rate:")
        for s in ser.head(5).index: print(f"    {s:40s}  {ser[s]:.1%}")
        print(f"  Bottom 5 by {ttype} rate:")
        for s in ser.tail(5).index: print(f"    {s:40s}  {ser[s]:.1%}")

if "recovery" in subj_traj_fr.columns and len(subj_traj_fr)>=10:
    rec_ser = subj_traj_fr["recovery"].sort_values(ascending=False)
    top5    = rec_ser.head(5).index.tolist()
    bot5    = rec_ser.tail(5).index.tolist()

    blank = pd.DataFrame({c:[0.0] for c in interesting}, index=[""])
    plot_data = pd.concat([subj_traj_fr.loc[top5,interesting], blank, subj_traj_fr.loc[bot5,interesting]])

    fig, ax = plt.subplots(figsize=(12, 7))
    plot_data.plot(kind="barh", stacked=True, ax=ax,
                   color=[TRAJ_COLORS.get(c,"gray") for c in interesting],
                   edgecolor="white", linewidth=0.3)
    n_rows = len(plot_data)
    ax.axhline(y=n_rows-5-0.5, color="black", linewidth=1.5, linestyle="--", alpha=0.7)
    for i, subj in enumerate(plot_data.index):
        if subj=="" or subj not in rec_ser.index: continue
        rv = rec_ser[subj]
        ax.text(rv+0.008, i, f"{rv:.1%}", va="center", fontsize=9, color="#2980b9", fontweight="bold")
    ax.text(1.02, 0.82, "Highest\nRecovery\n(self-corrects)", transform=ax.transAxes,
            fontsize=9, color="#27ae60", fontweight="bold", va="center")
    ax.text(1.02, 0.18, "Lowest\nRecovery\n(stays collapsed)", transform=ax.transAxes,
            fontsize=9, color="#c0392b", fontweight="bold", va="center")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title("Top 5 vs. Bottom 5 Subjects by Recovery Rate\n"
                 "Recovery = collapses at low k, then resists at higher k (self-inoculation)",
                 fontsize=12, fontweight="bold")
    ax.legend(title="Trajectory", bbox_to_anchor=(1.18,0.5), loc="center left", fontsize=9)
    savefig("A11_top_bottom_recovery.pdf")

In [ ]:
# ─── A12: Recovery deep-dive ──────────────────────────────────────────────────
print("\n[A12] Recovery Phenomenon Deep-Dive")
hline()
rec   = multi_k[multi_k["trajectory"]=="recovery"]
norec = multi_k[multi_k["trajectory"].isin(["immediate","threshold"])]
print(f"  Recovery trajectories: {len(rec):,}  ({len(rec)/len(multi_k)*100:.1f}% of multi-k)")
print(f"  Non-recovery collapses: {len(norec):,}")

COMP = [("bl_length","Baseline length"),("bl_hedge","Baseline hedge"),
        ("bl_confidence","Baseline confidence"),("co_length","Coercion length"),
        ("co_confidence","Coercion confidence"),("re_net","Response net signal")]

print("\n  Feature comparison (recovery vs non-recovery):")
for feat, label in COMP:
    a, b = norec[feat].dropna(), rec[feat].dropna()
    p = mwu_p(a, b)
    print(f"    {label:30s}  non-rec μ={a.mean():.3f}  rec μ={b.mean():.3f}  p={p:.4f} {sig(p)}")

if len(rec)>=5:
    # Self-undermining hypothesis
    rec_ids   = multi_k[multi_k["trajectory"]=="recovery"][GROUP].copy()
    rec_flat2 = succ_flat.merge(rec_ids, on=GROUP, how="inner")
    rec_onset = rec_flat2.merge(
        multi_k[multi_k["trajectory"]=="recovery"][GROUP+["onset_k"]], on=GROUP, how="inner")
    at_coll = rec_onset[rec_onset["k"]==rec_onset["onset_k"]]
    at_recv = rec_onset[(rec_onset["k"]>rec_onset["onset_k"]) & (~rec_onset["collapsed"])]
    print(f"\n  Self-undermining hypothesis (coercion argument at collapse vs recovery k):")
    print(f"    co_hedge at collapse k:    {at_coll['co_hedge'].mean():.4f}")
    print(f"    co_hedge at recovery k:    {at_recv['co_hedge'].mean():.4f}  p={mwu_p(at_coll['co_hedge'], at_recv['co_hedge']):.4f} {sig(mwu_p(at_coll['co_hedge'], at_recv['co_hedge']))}")
    print(f"    co_confidence at collapse: {at_coll['co_confidence'].mean():.4f}")
    print(f"    co_confidence at recovery: {at_recv['co_confidence'].mean():.4f}  p={mwu_p(at_coll['co_confidence'], at_recv['co_confidence']):.4f} {sig(mwu_p(at_coll['co_confidence'], at_recv['co_confidence']))}")
    print("    Interpretation: higher hedge / lower confidence at recovery k")
    print("    = longer argument exposes its own inconsistency = self-undermining confirmed")

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    af = axes.flatten()
    for ax, (feat, label) in zip(af, COMP):
        a, b = norec[feat].dropna(), rec[feat].dropna()
        p = mwu_p(a, b)
        bp = ax.boxplot([a, b], labels=["Non-recovery\ncollapse","Recovery"],
                        patch_artist=True, widths=0.5,
                        medianprops=dict(color="black",linewidth=2.5),
                        flierprops=dict(marker=".",markersize=3,alpha=0.4))
        bp["boxes"][0].set(facecolor="#c0392b", alpha=0.6)
        bp["boxes"][1].set(facecolor="#2980b9", alpha=0.6)
        ax.set_title(f"{label}\np={p:.3f} {sig(p)}", fontsize=9, fontweight="bold")
        for xi, vals in enumerate([a,b],1):
            ax.text(xi, ax.get_ylim()[1]*0.96, f"μ={vals.mean():.3f}",
                    ha="center", fontsize=8, color="navy", va="top", fontweight="bold")
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Recovery vs. Non-Recovery Collapse\n"
                 "What distinguishes samples that collapse at low k but self-correct at higher k?",
                 fontsize=13, fontweight="bold", y=1.02)
    savefig("A12_recovery_features.pdf")

In [ ]:
bcr_self

In [ ]:
# ─── A13: DVI vs BCR + Omega ─────────────────────────────────────────────────
print("\n[A13] Domain Vulnerability Index (DVI) vs BCR + Omega")
hline()
dvi = (
    crr_df.groupby("subject")
    .agg(total=("refused","count"), succeeded=("refused", lambda x: (~x).sum()))
    .assign(DVI=lambda d: d["succeeded"]/d["total"])
    .reset_index()
)
bcr_subj = (
    bcr_self.groupby("subject")
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(BCR=lambda d: d["collapsed"]/d["pool"]).reset_index()
)
dvi_bcr = dvi.merge(bcr_subj, on="subject").dropna()
rp, pp = pearsonr(dvi_bcr["DVI"], dvi_bcr["BCR"])
rs, ps = spearmanr(dvi_bcr["DVI"], dvi_bcr["BCR"])
print(f"  DVI vs BCR:  Pearson r={rp:.3f} p={pp:.4f}  Spearman r={rs:.3f} p={ps:.4f}")

# Omega
bcr_sm = (
    bcr_self.groupby(["model_short","subject"])["any_collapsed"]
    .mean().reset_index(name="BCR")
)
omega = (
    bcr_sm.groupby("model_short")["BCR"]
    .agg(Omega=np.var, mean_BCR="mean", n_subjects="count")
    .reset_index().sort_values("Omega",ascending=False)
)
print("\n  Domain Differentiation Score (Ω = variance of BCR across subjects):")
print(omega.round(4).to_string(index=False))
print("  High Ω = structured fragility (some domains protected, others not)")
print("  Low Ω  = uniform fragility (collapses everywhere equally)")

In [ ]:
import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams["figure.figsize"] = [8, 6]
import matplotlib.pyplot as plt
plt.close("all")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# Based on Hendrycks et al. (2021) categories.py, with Health split into Medical
subcategories = {
    "abstract_algebra": "math", "anatomy": "health", "astronomy": "physics",
    "business_ethics": "business", "clinical_knowledge": "health",
    "college_biology": "biology", "college_chemistry": "chemistry",
    "college_computer_science": "computer science", "college_mathematics": "math",
    "college_medicine": "health", "college_physics": "physics",
    "computer_security": "computer science", "conceptual_physics": "physics",
    "econometrics": "economics", "electrical_engineering": "engineering",
    "elementary_mathematics": "math", "formal_logic": "philosophy",
    "global_facts": "other", "high_school_biology": "biology",
    "high_school_chemistry": "chemistry", "high_school_computer_science": "computer science",
    "high_school_european_history": "history", "high_school_geography": "geography",
    "high_school_government_and_politics": "politics",
    "high_school_macroeconomics": "economics", "high_school_mathematics": "math",
    "high_school_microeconomics": "economics", "high_school_physics": "physics",
    "high_school_psychology": "psychology", "high_school_statistics": "math",
    "high_school_us_history": "history", "high_school_world_history": "history",
    "human_aging": "health", "human_sexuality": "culture",
    "international_law": "law", "jurisprudence": "law",
    "logical_fallacies": "philosophy", "machine_learning": "computer science",
    "management": "business", "marketing": "business",
    "medical_genetics": "health", "miscellaneous": "other",
    "moral_disputes": "philosophy", "moral_scenarios": "philosophy",
    "nutrition": "health", "philosophy": "philosophy",
    "prehistory": "history", "professional_accounting": "other",
    "professional_law": "law", "professional_medicine": "health",
    "professional_psychology": "psychology", "public_relations": "politics",
    "security_studies": "politics", "sociology": "culture",
    "us_foreign_policy": "politics", "virology": "health",
    "world_religions": "philosophy",
}

# Hendrycks original + Medical split from Other
categories = {
    "STEM":            ["physics", "chemistry", "biology", "computer science", "math", "engineering"],
    "Humanities":      ["history", "philosophy", "law"],
    "Social Sciences": ["politics", "culture", "economics", "geography", "psychology"],
    "Health":         ["health"],
    "Other":           ["other", "business"],
}

def get_category(subject):
    sub = subcategories.get(subject, "other")
    for cat, subs in categories.items():
        if sub in subs:
            return cat
    return "Other"

CATEGORY_COLORS = {
    "STEM":            "#4ECDA5",   # teal green 
    "Humanities":      "#F5A623",   # orange
    "Social Sciences": "#6BABDF",   # blue
    "Health":         "#E8668B",   # pink
    "Other":           "#C4956A",   # brown/tan
}

dvi_bcr["category"] = dvi_bcr["subject"].map(get_category)

fig, ax = plt.subplots(figsize=(8, 6))

for cat, color in CATEGORY_COLORS.items():
    mask = dvi_bcr["category"] == cat
    if mask.sum() == 0:
        continue
    ax.scatter(dvi_bcr.loc[mask, "DVI"], dvi_bcr.loc[mask, "BCR"],
               s=dvi_bcr.loc[mask, "pool"].clip(20, 500),
               color=color, edgecolors="#2D2D2D", linewidth=0.5,
               alpha=0.65, zorder=4, label=cat)

# Trend line
z = np.polyfit(dvi_bcr["DVI"], dvi_bcr["BCR"], 1)
xr = np.linspace(dvi_bcr["DVI"].min(), dvi_bcr["DVI"].max(), 100)
ax.plot(xr, np.poly1d(z)(xr), "--", color="#2D2D2D", linewidth=1.5, alpha=0.5)

# Labels — top 7 and bottom 7
top7 = dvi_bcr.nlargest(7, "BCR")
bot7 = dvi_bcr.nsmallest(7, "BCR")

offsets_top = [(8, 6), (8, -10), (-40, 8), (8, -12), (-40, -8), (8, 10), (-40, -12)]
offsets_bot = [(8, -8), (8, 6), (-40, -10), (8, 8), (-40, 6), (8, -12), (-40, 10)]

for i, (_, row) in enumerate(top7.iterrows()):
    ax.annotate(row["subject"], (row["DVI"], row["BCR"]),
                fontsize=8, color="#8B2500",
                xytext=offsets_top[i % len(offsets_top)], textcoords="offset points",
                arrowprops=dict(arrowstyle="-", color="#999999", linewidth=0.5))
for i, (_, row) in enumerate(bot7.iterrows()):
    ax.annotate(row["subject"], (row["DVI"], row["BCR"]),
                fontsize=8, color="#1A6B3C",
                xytext=offsets_bot[i % len(offsets_bot)], textcoords="offset points",
                arrowprops=dict(arrowstyle="-", color="#999999", linewidth=0.5))

ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("CSR (coercion success rate)", fontsize=11)
ax.set_ylabel("AFR", fontsize=11)
ax.tick_params(axis="both", labelsize=10)
ax.grid(alpha=0.2, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=9, frameon=False, loc="upper left")

fig.tight_layout()
fig.savefig("figures/A13_CSR_AFR_Omega.pdf", dpi=150, bbox_inches="tight")
plt.close("all")
print("  [fig] saved")

In [ ]:
2052 / 57

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK B — CROSS REPO
# ══════════════════════════════════════════════════════════════════════════════

section("BLOCK B — CROSS-MODEL ANALYSIS")

# ─── B1: Cross-BCR matrix ─────────────────────────────────────────────────────
print("\n[B1] Cross-BCR N×N Matrix")
hline()
cross_agg = (
    bcr_cross.groupby(["cm_short","bm_short"])
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(CMCR=lambda d: d["collapsed"]/d["pool"])
    .reset_index()
)
print(cross_agg.to_string(index=False))

self_bcr_ref = {
    m: v for m, v in (
        bcr_self[bcr_self["attribution"]=="blind"]
        .groupby("model_short")["any_collapsed"].mean().to_dict().items()
    )
}
all_models_cross = sorted(set(cross_agg["cm_short"]) | set(cross_agg["bm_short"]))
matrix = pd.DataFrame(np.nan, index=all_models_cross, columns=all_models_cross)
for _, row in cross_agg.iterrows():
    matrix.loc[row["cm_short"], row["bm_short"]] = row["CMCR"]
for m in all_models_cross:
    if m in self_bcr_ref:
        matrix.loc[m, m] = self_bcr_ref[m]

print("\n  Cross-BCR matrix (row=coercion source, col=baseline model):")
print(matrix.round(3).to_string())


In [ ]:
# FIG B1: Cross-BCR heatmap with annotations
if not matrix.empty and not matrix.isna().all().all():
    fig, ax = plt.subplots(figsize=(max(9, len(all_models_cross)*1.5),
                                    max(7, len(all_models_cross)*1.3)))
    # Create annotation matrix with special diagonal marker
    annot_matrix = matrix.copy().astype(object)
    for m in all_models_cross:
        if not np.isnan(matrix.loc[m,m]):
            annot_matrix.loc[m,m] = f"[{matrix.loc[m,m]:.0%}]\n(self)"

    mask = matrix.isna()
    sns.heatmap(matrix.astype(float), annot=True, fmt=".0%", cmap="RdYlGn_r",
                vmin=0, vmax=1, ax=ax, mask=mask,
                linewidths=1.2, linecolor="white",
                cbar_kws={"label":"Collapse Rate","shrink":0.7},
                annot_kws={"size":10,"weight":"bold"})
    # Highlight diagonal
    for i, m in enumerate(all_models_cross):
        ax.add_patch(plt.Rectangle((i,i), 1, 1, fill=False,
                                   edgecolor="navy", lw=3, zorder=5))
    ax.set_title("Cross-Model Belief Collapse Rate Matrix\n"
                 "Row = coercion source (whose wrong reasoning is used)\n"
                 "Col = baseline model (who is being challenged)\n"
                 "[X%] on diagonal = self-BCR",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Baseline model (being challenged)", fontsize=11)
    ax.set_ylabel("Coercion model (reasoning source)", fontsize=11)
    plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
    savefig("B1_cross_bcr_matrix.pdf")


In [ ]:

# ─── B2: Self vs. cross comparison ────────────────────────────────────────────
print("\n[B2] Self vs. Cross BCR Comparison")
hline()
# For each baseline model: self-BCR vs mean cross-BCR from other models
cross_by_bm = (
    bcr_cross[bcr_cross["cm_short"]!=bcr_cross["bm_short"]]
    .groupby("bm_short")["any_collapsed"].mean()
    .reset_index().rename(columns={"bm_short":"model","any_collapsed":"cross_BCR"})
)
self_df = pd.DataFrame([{"model":m,"self_BCR":v} for m,v in self_bcr_ref.items()])
comp_df = self_df.merge(cross_by_bm, on="model", how="inner")
comp_df["delta"] = comp_df["cross_BCR"] - comp_df["self_BCR"]
print(comp_df.round(4).to_string(index=False))
print("\n  Positive delta = another model's wrong reasoning is MORE effective than your own")
print("  Negative delta = your own wrong reasoning is MORE effective against you")

if len(comp_df)>=2:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(comp_df)); w=0.35
    ax.bar(x-w/2, comp_df["self_BCR"],  w, label="Self-BCR\n(own coercion)", color="#8e44ad", edgecolor="white", alpha=0.85)
    ax.bar(x+w/2, comp_df["cross_BCR"], w, label="Cross-BCR\n(peer coercion)", color="#e67e22", edgecolor="white", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(comp_df["model"], rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title("Self-BCR vs. Cross-BCR: Is Your Own Reasoning More Dangerous?\n"
                 "Δ = cross − self: positive = peer reasoning more effective",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    for i, row in comp_df.iterrows():
        c = "#c0392b" if row["delta"]>0 else "#27ae60"
        ax.text(i, max(row["self_BCR"],row["cross_BCR"])+0.018,
                f"Δ={row['delta']:+.1%}", ha="center", fontsize=9, color=c, fontweight="bold")
    savefig("B2_self_vs_cross.pdf")


In [ ]:

# ─── B3: EP, EA, APD ─────────────────────────────────────────────────────────
print("\n[B3] Epistemic Porosity (EP), Authority (EA), APD")
hline()
ep_ea = []
for model in all_models_cross:
    ep_vals = matrix.loc[[m for m in all_models_cross if m!=model], model].dropna()
    ea_vals = matrix.loc[model, [m for m in all_models_cross if m!=model]].dropna()
    ep_ea.append({"model":model,
                  "EP": ep_vals.mean() if len(ep_vals) else np.nan,
                  "EA": ea_vals.mean() if len(ea_vals) else np.nan,
                  "self_BCR": self_bcr_ref.get(model, np.nan)})
ep_df = pd.DataFrame(ep_ea).dropna(subset=["EP","EA"])
ep_df["APD"] = ep_df["EA"] - ep_df["EP"]
ep_df = ep_df.sort_values("APD", ascending=False)
print(ep_df.round(4).to_string(index=False))
print("\n  EP  = mean cross-BCR when THIS model is the target (how easily destabilized)")
print("  EA  = mean cross-BCR THIS model causes in others (how persuasive even when wrong)")
print("  APD = EA − EP: positive = net epistemic exporter (good debate anchor)")
print("                 negative = net epistemic importer (avoid as anchor)")


In [ ]:
if len(ep_df) >= 2:
    import matplotlib
    matplotlib.rcParams.update(matplotlib.rcParamsDefault)
    import matplotlib.pyplot as plt
    plt.close("all")
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "font.size": 10,
        "text.color": "black",
        "axes.labelcolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
    })

    LABEL_OFFSETS = {
        "GPT-5.1":       (-55, +10),
        "Qwen3.5-35B":   (-60, -15),
        "Gemma-4-26B":   (10, 10),
        "Qwen3.5-4B":    (12, 12),
        "Qwen3.5-9B":    (-50, -20),
        "Llama-3.3-70B": (-25, +15),
        "Llama-3.1-8B":  (-50, 10),
    }

    fig, ax = plt.subplots(figsize=(7, 5.4))

    for _, row in ep_df.iterrows():
        ax.scatter(row["EP"], row["EA"], s=150, zorder=5, edgecolors="#2D2D2D", linewidth=0.8,
                   color=PALETTE[list(ep_df["model"]).index(row["model"]) % len(PALETTE)])
        ox, oy = LABEL_OFFSETS.get(row["model"], (12, 6))
        ax.annotate(row["model"], (row["EP"], row["EA"]),
                    xytext=(ox, oy), textcoords="offset points",
                    fontsize=9, fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color="#999999", linewidth=0.5),
                    zorder=6)

    lim = [0, max(ep_df[["EP", "EA"]].max()) * 1.1]
    ax.plot(lim, lim, "k--", linewidth=1.2, alpha=0.4, label="EA = EP")
    ax.fill_between(lim, lim, lim[1], alpha=0.06, color="green", label="EA > EP (exporter)")
    ax.fill_between(lim, 0, lim, alpha=0.06, color="red", label="EP > EA (importer)")

    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Epistemic Porosity (EP)\nhow easily the model is destabilized by others", fontsize=12)
    ax.set_ylabel("Epistemic Authority (EA)\nhow persuasive the model's wrong reasoning is", fontsize=12)
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(alpha=0.2, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=10, frameon=True, loc="upper left")

    fig.tight_layout()
    fig.savefig("figures/B3_EP_EA_APD.pdf", dpi=150, bbox_inches="tight")
    plt.close("all")
    print("  [fig] saved")

In [ ]:
# ─── B4: Cross-BCR by subject ─────────────────────────────────────────────────
print("\n[B4] Cross-BCR by Domain")
hline()
cross_subj = (
    bcr_cross.groupby(["subject","cm_short","bm_short"])["any_collapsed"]
    .mean().reset_index(name="CMCR")
)
cross_subj_agg = cross_subj.groupby("subject")["CMCR"].mean().reset_index(name="mean_CMCR")
subj_comp = self_subj_bcr = (
    bcr_self.groupby("subject")
    .agg(pool=("any_collapsed","count"), collapsed=("any_collapsed","sum"))
    .assign(BCR=lambda d: d["collapsed"]/d["pool"]).reset_index()
).merge(cross_subj_agg, on="subject", how="inner")

if len(subj_comp)>=4:
    rs, ps = spearmanr(subj_comp["BCR"], subj_comp["mean_CMCR"])
    print(f"  Self-BCR vs mean cross-BCR: Spearman r={rs:.3f} p={ps:.4f}")

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(subj_comp["BCR"], subj_comp["mean_CMCR"],
               s=subj_comp["pool"].clip(20,400), alpha=0.7, edgecolors="gray", linewidth=0.5, zorder=4)
    for _, row in subj_comp.nlargest(6,"BCR").iterrows():
        ax.annotate(row["subject"], (row["BCR"],row["mean_CMCR"]),
                    fontsize=6, xytext=(3,2), textcoords="offset points", color="#c0392b")
    ax.plot([0,1],[0,1],"k--",linewidth=1.2,alpha=0.4,label="self=cross")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Self-BCR per subject"); ax.set_ylabel("Mean cross-BCR per subject")
    ax.set_title(f"Self vs. Cross Vulnerability by Domain\nSpearman r={rs:.3f}  (bubble=pool size)",
                 fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3); ax.legend()
    savefig("B4_cross_domain.pdf")

# ─── B5: Self vs Other argument effectiveness — THE KEY CROSS FIGURE ──────────
print("\n[B5] Self vs. Other Argument: Which is More Effective? (Cross-model heat)")
hline()
# For each baseline model b, compare:
#   self-BCR (model b challenged by b's own coercion)
#   cross-BCR from each challenger model a→b
# Then also show the full pairwise matrix as a proper heatmap

if not matrix.empty and not matrix.isna().all().all():
    # Row-normalize: for each baseline model (column), show how much each
    # challenger exceeds the self-BCR
    base_models = [m for m in all_models_cross if m in matrix.columns]
    delta_matrix = matrix.copy()
    for bm in base_models:
        self_val = matrix.loc[bm, bm] if not np.isnan(matrix.loc[bm, bm]) else np.nan
        delta_matrix[bm] = matrix[bm] - self_val

    print("  Δ matrix (CMCR − self-BCR per baseline model):")
    print("  Positive = challenger's wrong reasoning MORE effective than model's own")
    print("  Negative = model's own wrong reasoning MORE effective against itself")
    print(delta_matrix.round(3).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(18, max(7, len(all_models_cross)*1.2)))

    # Left: raw matrix
    ax = axes[0]
    mask_l = matrix.isna()
    sns.heatmap(matrix.astype(float), annot=True, fmt=".0%",
                cmap="YlOrRd", vmin=0, vmax=1, ax=ax, mask=mask_l,
                linewidths=1, linecolor="white",
                cbar_kws={"label":"Collapse Rate"},
                annot_kws={"size":10,"weight":"bold"})
    for i in range(len(all_models_cross)):
        ax.add_patch(plt.Rectangle((i,i),1,1,fill=False,edgecolor="navy",lw=3,zorder=5))
    ax.set_title("Cross-BCR Matrix\n[Diagonal = self-BCR]",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Baseline model (challenged)"); ax.set_ylabel("Coercion model (source)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)

    # Right: delta matrix (crosses − self)
    ax = axes[1]
    vabs = max(abs(delta_matrix.values[~np.isnan(delta_matrix.values)].max()),
               abs(delta_matrix.values[~np.isnan(delta_matrix.values)].min()))
    mask_r = delta_matrix.isna()
    # Fill diagonal with 0 for display
    delta_disp = delta_matrix.fillna(0)
    for i, m in enumerate(all_models_cross):
        delta_disp.loc[m, m] = 0  # diagonal: self vs self = 0 by definition
    sns.heatmap(delta_disp.astype(float), annot=True, fmt=".0%",
                cmap="RdBu_r", vmin=-vabs, vmax=vabs, ax=ax,
                linewidths=1, linecolor="white",
                cbar_kws={"label":"Δ vs. self-BCR"},
                annot_kws={"size":10,"weight":"bold"})
    for i in range(len(all_models_cross)):
        ax.add_patch(plt.Rectangle((i,i),1,1,fill=False,edgecolor="gray",lw=2,
                                   linestyle="--",zorder=5))
    ax.set_title("Δ = CMCR(A→B) − self-BCR(B)\nRed = peer more effective | Blue = self more effective",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Baseline model (challenged)"); ax.set_ylabel("Coercion model (source)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)

    fig.suptitle("Is Another Model's Wrong Reasoning More Effective Than Your Own?\n"
                 "Left: absolute collapse rates | Right: difference vs. self-challenge",
                 fontsize=13, fontweight="bold", y=1.02)
    savefig("B5_self_vs_other_heatmap.pdf")

# ─── B6: Cross-argument features ──────────────────────────────────────────────
print("\n[B6] Coercion Argument Features vs. Cross-Collapse")
hline()
CROSS_FEATS = [("co_confidence","Coercion confidence"),("co_hedge","Coercion hedge"),
               ("co_length","Coercion length"),("re_net","Response net signal")]
print("  Feature comparison (collapsed vs held in cross challenges):")
for feat, label in CROSS_FEATS:
    if feat not in flat_cross.columns: continue
    a = flat_cross[flat_cross["collapsed"]==True][feat].dropna()
    b = flat_cross[flat_cross["collapsed"]==False][feat].dropna()
    p = mwu_p(a, b)
    print(f"    {label:30s}  collapsed μ={a.mean():.3f}  held μ={b.mean():.3f}  p={p:.4f} {sig(p)}")

if len(flat_cross)>=20:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    af = axes.flatten()
    for ax, (feat, label) in zip(af, CROSS_FEATS):
        if feat not in flat_cross.columns: continue
        a = flat_cross[flat_cross["collapsed"]==False][feat].dropna()
        b = flat_cross[flat_cross["collapsed"]==True][feat].dropna()
        p = mwu_p(a, b)
        bp = ax.boxplot([a, b], labels=["Held","Collapsed"],
                        patch_artist=True, widths=0.5,
                        medianprops=dict(color="black",linewidth=2.5),
                        flierprops=dict(marker=".",markersize=3,alpha=0.4))
        bp["boxes"][0].set(facecolor="#2980b9", alpha=0.6)
        bp["boxes"][1].set(facecolor="#c0392b", alpha=0.6)
        ax.set_title(f"{label}\np={p:.3f} {sig(p)}", fontsize=10, fontweight="bold")
        for xi, vals in enumerate([a,b],1):
            ax.text(xi, ax.get_ylim()[1]*0.96, f"μ={vals.mean():.3f}",
                    ha="center", fontsize=9, color="navy", va="top", fontweight="bold")
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Cross-Model: Does Coercion Argument Quality Drive Collapse?\n"
                 "Does model A's argument style predict collapse in model B?",
                 fontsize=13, fontweight="bold", y=1.02)
    savefig("B6_cross_argument_features.pdf")

# ══════════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

section("FINAL SUMMARY — ALL KEY NUMBERS")

print("\n[SELF REPO] BCR overview (blind, all k):")
print(bcr_blind_agg.to_string(index=False))

print("\n[SELF REPO] Trajectory distribution (multi-k):")
tv2 = multi_k["trajectory"].value_counts()
for t,c in tv2.items():
    print(f"  {t:<14}: {c:>6,}  ({c/len(multi_k)*100:.1f}%)")

print("\n[SELF REPO] CRR (coercion resistance):")
print(crr_model[["model_short","CRR","total"]].to_string(index=False))

print("\n[SELF REPO] RSS (metacognitive awareness):")
print(rss_df[["model_short","RSS","CRR_correct","CRR_incorrect"]].to_string(index=False))

print("\n[SELF REPO] Omega (domain differentiation):")
print(omega.to_string(index=False))

print("\n[CROSS REPO] Cross-BCR matrix:")
print(matrix.apply(lambda col: col.map(lambda x: f"{x:.1%}" if not np.isnan(x) else "—")).to_string())

print("\n[CROSS REPO] EP / EA / APD:")
print(ep_df[["model","EP","EA","APD","self_BCR"]].round(4).to_string(index=False))

if "delta" in comp_df.columns:
    print("\n[CROSS REPO] Self vs cross comparison:")
    print(comp_df.round(4).to_string(index=False))

all_figs = sorted(os.listdir(OUT_DIR))
print(f"\n{'─'*65}")
print(f"  {len(all_figs)} figures saved to ./{OUT_DIR}/")
for f in all_figs: print(f"    {f}")
print("Done.")

In [ ]:
import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
import matplotlib.pyplot as plt
import numpy as np
plt.close("all")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# Build matrices from ep_df / cross data
models = ["GPT-5.1", "Gemma-4-26B", "Llama-3.1-8B", "Llama-3.3-70B",
          "Qwen3.5-35B", "Qwen3.5-4B", "Qwen3.5-9B"]

# Absolute CMCR matrix (source → challenged)
cmcr = np.array([
    [0.213, 0.298, 1.000, 0.938, 0.224, 0.706, 0.616],
    [0.172, 0.207, 0.996, 0.890, 0.164, 0.725, 0.543],
    [0.111, 0.057, 0.968, 0.570, 0.059, 0.362, 0.254],
    [0.156, 0.139, 0.994, 0.793, 0.107, 0.556, 0.441],
    [0.173, 0.208, 0.984, 0.866, 0.157, 0.708, 0.511],
    [0.158, 0.172, 0.975, 0.848, 0.138, 0.719, 0.508],
    [0.150, 0.175, 0.993, 0.820, 0.107, 0.575, 0.458],
])

# Self-BCR (diagonal)
self_bcr = np.diag(cmcr)

# Delta matrix
delta = cmcr - self_bcr[np.newaxis, :]

fig, ax = plt.subplots(figsize=(8, 7))

# Use diverging colormap: blue (self more effective) to red (peer more effective)
from matplotlib.colors import TwoSlopeNorm
norm = TwoSlopeNorm(vmin=-0.30, vcenter=0, vmax=0.25)
im = ax.imshow(delta, cmap="PuOr_r", norm=norm, aspect="equal")

# Add text annotations: CMCR / delta
for i in range(len(models)):
    for j in range(len(models)):
        abs_val = cmcr[i, j]
        d_val = delta[i, j]
        if i == j:
            # Diagonal: bold self-BCR
            ax.text(j, i, f"{abs_val:.0%}", ha="center", va="center",
                    fontsize=9, fontweight="bold", color="black",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#2D2D2D", linewidth=1))
        else:
            # Off-diagonal: CMCR on top, delta below
            text_color = "white" if abs(d_val) > 0.15 else "black"
            ax.text(j, i - 0.12, f"{abs_val:.0%}", ha="center", va="center",
                    fontsize=8, fontweight="bold", color=text_color)
            ax.text(j, i + 0.18, f"{d_val:+.0%}", ha="center", va="center",
                    fontsize=7, color=text_color, style="italic")

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=9)
ax.set_xlabel("Baseline model (challenged)", fontsize=11)
ax.set_ylabel("Coercion model (source)", fontsize=11)

# Colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label("$\\Delta$ vs. self-AFR", fontsize=10)

# Highlight diagonal
for i in range(len(models)):
    ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                                fill=False, edgecolor="#2D2D2D", linewidth=2))

ax.spines[["top", "right", "bottom", "left"]].set_visible(False)

fig.tight_layout()
fig.savefig("figures/B5_cross_matrix.pdf", dpi=150, bbox_inches="tight")
plt.close("all")
print("  [fig] saved")

In [ ]:
cross_by_subj = (
    flat_cross[flat_cross["coercion_k"] == 10]
    .groupby(["cm_short", "bm_short", "original_id", "subject"])
    .agg(collapsed=("collapsed", "any"))  # question-level: flipped if any option did
    .reset_index()
    .groupby(["cm_short", "bm_short", "subject"])
    .agg(cmfr=("collapsed", "mean"))
    .reset_index()
)

grand_mean = cross_by_subj["cmfr"].mean()
ss_total = ((cross_by_subj["cmfr"] - grand_mean) ** 2).sum()

subject_means  = cross_by_subj.groupby("subject")["cmfr"].mean()
source_means   = cross_by_subj.groupby("cm_short")["cmfr"].mean()
baseline_means = cross_by_subj.groupby("bm_short")["cmfr"].mean()

def ss_between(means, counts):
    return sum(counts[g] * (means[g] - grand_mean) ** 2 for g in means.index)

ss_subj = ss_between(subject_means,  cross_by_subj.groupby("subject").size())
ss_src  = ss_between(source_means,   cross_by_subj.groupby("cm_short").size())
ss_bsl  = ss_between(baseline_means, cross_by_subj.groupby("bm_short").size())
ss_resid = ss_total - ss_subj - ss_src - ss_bsl

print(f"Baseline: {ss_bsl/ss_total:.1%}")
print(f"Source:   {ss_src/ss_total:.1%}")
print(f"Subject:  {ss_subj/ss_total:.1%}")
print(f"Residual: {ss_resid/ss_total:.1%}")

In [ ]:
# ─── Option 1: Per-baseline model rank correlation ───────────────────────────
print("\n[B7] Per-baseline subject rank preservation across challengers")
hline()

models_cross = sorted(flat_cross["bm_short"].unique())
correlations = []

for bm in models_cross:
    # Self-source subject ranking for this baseline
    self_subj = (
        succ_flat[(succ_flat["model_short"] == bm) & (succ_flat["attribution"] == "blind")]
        .groupby("subject")["collapsed"].mean()
    )
    
    for cm in models_cross:
        if cm == bm:
            continue
        # Cross-source subject ranking
        cross_subj = (
            flat_cross[(flat_cross["bm_short"] == bm) & (flat_cross["cm_short"] == cm)]
            .groupby("subject")["collapsed"].mean()
        )
        
        # Align on shared subjects
        shared = self_subj.index.intersection(cross_subj.index)
        if len(shared) < 10:
            continue
        r, p = spearmanr(self_subj[shared], cross_subj[shared])
        correlations.append({
            "baseline": bm, "source": cm, "r": r, "p": p, "n_subjects": len(shared)
        })

corr_df = pd.DataFrame(correlations)
if len(corr_df) > 0:
    print(f"  Pairwise Spearman r (subject ranking: self-source vs cross-source):")
    print(f"    Median r: {corr_df['r'].median():.3f}")
    print(f"    Mean r:   {corr_df['r'].mean():.3f}")
    print(f"    Min r:    {corr_df['r'].min():.3f}")
    print(f"    Max r:    {corr_df['r'].max():.3f}")
    print(f"    Total pairs: {len(corr_df)}")
    
    print(f"\n  Per baseline model:")
    for bm in models_cross:
        sub = corr_df[corr_df["baseline"] == bm]
        if len(sub) == 0:
            continue
        print(f"    {bm:20s}: median r={sub['r'].median():.3f}  "
              f"range [{sub['r'].min():.3f}, {sub['r'].max():.3f}]  "
              f"n_pairs={len(sub)}")
    
    print(f"\n  Per source model:")
    for cm in models_cross:
        sub = corr_df[corr_df["source"] == cm]
        if len(sub) == 0:
            continue
        print(f"    {cm:20s}: median r={sub['r'].median():.3f}  "
              f"range [{sub['r'].min():.3f}, {sub['r'].max():.3f}]  "
              f"n_pairs={len(sub)}")
    
    # Full pairwise table
    print(f"\n  Full pairwise Spearman r table:")
    r_matrix = corr_df.pivot(index="source", columns="baseline", values="r")
    print(r_matrix.round(3).to_string())


# ─── Option 2: Variance decomposition ────────────────────────────────────────
print("\n[B8] Variance decomposition: subject vs source vs baseline")
hline()

cross_by_subj = (
    flat_cross.groupby(["cm_short", "bm_short", "subject"]).agg(
        bcr=("collapsed", "mean"),
        n=("collapsed", "count")
    ).reset_index()
)

# Manual sum-of-squares decomposition (avoids statsmodels dependency)
grand_mean = cross_by_subj["bcr"].mean()

# SS for each factor
ss_subject = cross_by_subj.groupby("subject")["bcr"].transform("mean")
ss_source = cross_by_subj.groupby("cm_short")["bcr"].transform("mean")
ss_baseline = cross_by_subj.groupby("bm_short")["bcr"].transform("mean")

n = len(cross_by_subj)
ss_total = ((cross_by_subj["bcr"] - grand_mean) ** 2).sum()

# Between-group SS for each factor
subject_means = cross_by_subj.groupby("subject")["bcr"].mean()
source_means = cross_by_subj.groupby("cm_short")["bcr"].mean()
baseline_means = cross_by_subj.groupby("bm_short")["bcr"].mean()

ss_subj = sum(
    cross_by_subj.groupby("subject").size()[s] * (subject_means[s] - grand_mean) ** 2
    for s in subject_means.index
)
ss_src = sum(
    cross_by_subj.groupby("cm_short").size()[s] * (source_means[s] - grand_mean) ** 2
    for s in source_means.index
)
ss_bsl = sum(
    cross_by_subj.groupby("bm_short").size()[s] * (baseline_means[s] - grand_mean) ** 2
    for s in baseline_means.index
)
ss_resid = ss_total - ss_subj - ss_src - ss_bsl

print(f"  Grand mean BCR: {grand_mean:.3f}")
print(f"  Total SS: {ss_total:.4f}")
print(f"\n  Variance explained (Type I, additive):")
print(f"    Subject (which topic):   {ss_subj/ss_total:6.1%}  (SS={ss_subj:.4f})")
print(f"    Source (who authored):    {ss_src/ss_total:6.1%}  (SS={ss_src:.4f})")
print(f"    Baseline (who is challenged): {ss_bsl/ss_total:6.1%}  (SS={ss_bsl:.4f})")
print(f"    Residual:                {ss_resid/ss_total:6.1%}  (SS={ss_resid:.4f})")
print(f"\n  Interpretation:")
if ss_subj > ss_src and ss_subj > ss_bsl:
    print(f"    Subject explains the most variance ({ss_subj/ss_total:.1%}) — content dominates.")
elif ss_bsl > ss_src:
    print(f"    Baseline model explains the most variance ({ss_bsl/ss_total:.1%}) — susceptibility dominates.")
else:
    print(f"    Source model explains the most variance ({ss_src/ss_total:.1%}) — authorship dominates.")
print(f"    Source model explains {ss_src/ss_total:.1%} — "
      f"{'minimal' if ss_src/ss_total < 0.10 else 'moderate' if ss_src/ss_total < 0.25 else 'substantial'} "
      f"contribution of who authored the argument.")

In [ ]:
"""
Robustness / Uncertainty Add-On
================================
Append this AFTER the existing analysis script (it consumes `flat`, `flat_cross`,
`bcr_self`, `crr_df`, `multi_k`, `succ_flat`, `MODEL_SHORT`, `OUT_DIR`, ...).

Addresses reviewer concerns about uncertainty quantification for:
  R1  Per-model AFR (Table 2)               → cluster bootstrap CIs over items
  R2  Self-Attribution Delta (Table 3)      → paired bootstrap + sign test + GLMM
  R3  Subject-level AFR spread (Table 5)    → subject CIs + permutation test for spread
  R4  Variance decomposition 77.7/13.3/8.1  → bootstrap CIs over (cm,bm,subject) cells
  R5  Cross vs self ∆ per model (Table 6)   → paired item bootstrap, FDR-corrected
  R6  MAXFLIP ∆ (Table 7)                   → paired bootstrap

All outputs go to OUT_DIR/robustness_*.{csv,tex,pdf}.
"""
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
N_BOOT       = 2000     # bootstrap replicates (drop to 500 for a quick smoke test)
ALPHA        = 0.05     # 95% CIs
RNG          = np.random.default_rng(20260503)
ROB_DIR      = os.path.join(OUT_DIR, "robustness")
os.makedirs(ROB_DIR, exist_ok=True)

def section_h(title):
    bar = "═" * 72
    print(f"\n{bar}\n  {title}\n{bar}")

def ci_pct(arr, alpha=ALPHA):
    """Percentile CI; returns (lo, hi) on the same scale as arr."""
    lo = np.nanpercentile(arr, 100 * alpha / 2)
    hi = np.nanpercentile(arr, 100 * (1 - alpha / 2))
    return lo, hi

def fmt_ci(lo, hi, scale=100, prec=1):
    return f"[{lo*scale:.{prec}f}, {hi*scale:.{prec}f}]"

# ─────────────────────────────────────────────────────────────────────────────
# QUESTION-LEVEL AFR POOL (matches paper Definition 3.1)
# ─────────────────────────────────────────────────────────────────────────────
#   Paper: AFR_c(k) = Pr[â_final ≠ a* | â_nat = a*, R(q,x,k) exists]
#   Denominator gate is per-QUESTION ("at least one wrong-option argument exists"),
#   not per-option, and a question counts as flipped if ANY wrong option flips it.
#
# The main-script `bcr_self` already implements this gate via `any_coercion_success`
# on the full row (one row per question×model×k×attribution). We use it directly
# and define `flipped = any_collapsed`. This reproduces Table 2 to within rounding.
# ─────────────────────────────────────────────────────────────────────────────
self_pool_cols = ["original_id", "subject", "model_short", "attribution",
                  "k_sentences", "any_collapsed", "baseline_correct",
                  "any_coercion_success"]
if "pressure" in bcr_self.columns:
    self_pool_cols.append("pressure")
self_pool = bcr_self[
    bcr_self["baseline_correct"] & bcr_self["any_coercion_success"]
    & ~bcr_self["k_sentences"].isin(EXCLUDED_K)
][self_pool_cols].copy()
self_pool = self_pool.rename(columns={"k_sentences": "k"})
self_pool["flipped"] = self_pool["any_collapsed"].astype(int)
print(f"Self-repo question-level pool (BC ∩ any-CS): {len(self_pool):,} rows  "
      f"(unit = question × model × attribution × k)")

# Choose pressure if available; paper uses 'high' for the headline
if "pressure" in self_pool.columns and "high" in self_pool["pressure"].unique():
    self_pool = self_pool[self_pool["pressure"] == "high"].copy()
    print(f"  After restricting to pressure='high': {len(self_pool):,} rows")


# ═════════════════════════════════════════════════════════════════════════════
#  R1 — Per-model AFR with cluster-bootstrap CIs (clusters = original_id)
#  Resamples whole questions. This is the right unit because the 3 wrong-option
#  rows for one question share the question text and the model's prior on it.
# ═════════════════════════════════════════════════════════════════════════════

section_h("R1  Per-model AFR — cluster bootstrap CIs (Table 2 augmentation)")

def cluster_bootstrap_rate(df, value_col, cluster_col="original_id",
                            n_boot=N_BOOT, rng=RNG):
    """Bootstrap a binary-mean by resampling clusters with replacement.
    Returns (point, lo, hi, n_clusters)."""
    if len(df) == 0:
        return np.nan, np.nan, np.nan, 0
    # Pre-group: arrays of values per cluster
    grouped = df.groupby(cluster_col)[value_col].apply(np.asarray)
    cluster_ids = grouped.index.to_numpy()
    cluster_vals = grouped.to_numpy(dtype=object)
    n_c = len(cluster_ids)
    point = df[value_col].mean()
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n_c, size=n_c)
        # Concatenate sampled cluster arrays
        sampled = np.concatenate([cluster_vals[i] for i in idx])
        boots[b] = sampled.mean()
    lo, hi = ci_pct(boots)
    return point, lo, hi, n_c

# Headline AFR (blind, all k pooled) per model — Table 2 'Mean' column
print("\n[R1a] AFR_blind per model (all k pooled) — clustered on original_id")
rows = []
blind = self_pool[self_pool["attribution"] == "blind"]
for m in sorted(blind["model_short"].unique()):
    sub = blind[blind["model_short"] == m]
    p, lo, hi, n_c = cluster_bootstrap_rate(sub, "flipped")
    rows.append({"model": m, "n_rows": len(sub), "n_questions": n_c,
                 "AFR": p, "ci_lo": lo, "ci_hi": hi,
                 "halfwidth_pp": (hi - lo) / 2 * 100})
afr_blind_ci = pd.DataFrame(rows).sort_values("AFR", ascending=False)
print(afr_blind_ci.assign(
    AFR=lambda d: (d["AFR"] * 100).round(1),
    ci=lambda d: [fmt_ci(l, h) for l, h in zip(d["ci_lo"], d["ci_hi"])],
    halfwidth_pp=lambda d: d["halfwidth_pp"].round(2),
)[["model", "n_questions", "AFR", "ci", "halfwidth_pp"]].to_string(index=False))

# Per-model × k AFR (filling out Table 2 with CIs)
print("\n[R1b] AFR_blind per model × k — clustered on original_id")
rows = []
for m in sorted(blind["model_short"].unique()):
    for k in sorted(blind["k"].unique()):
        sub = blind[(blind["model_short"] == m) & (blind["k"] == k)]
        if len(sub) == 0:
            continue
        p, lo, hi, n_c = cluster_bootstrap_rate(sub, "flipped")
        rows.append({"model": m, "k": int(k), "n": len(sub),
                     "AFR": p, "ci_lo": lo, "ci_hi": hi})
afr_mk_ci = pd.DataFrame(rows)
afr_mk_ci.to_csv(os.path.join(ROB_DIR, "R1_afr_blind_model_x_k.csv"), index=False)
print(afr_mk_ci.assign(
    AFR=lambda d: (d["AFR"] * 100).round(1),
    ci=lambda d: [fmt_ci(l, h) for l, h in zip(d["ci_lo"], d["ci_hi"])],
)[["model", "k", "n", "AFR", "ci"]].to_string(index=False))


# ═════════════════════════════════════════════════════════════════════════════
#  R2 — Self-Attribution Delta (Table 3)
#  THIS IS THE PAPER'S STRONGEST CLAIM ("positive for every model").
#  Two complementary tests:
#    (a) PAIRED bootstrap on the per-item difference flipped_self − flipped_blind
#        Items pair on (original_id, k) — question-level (paper Definition 3.1).
#    (b) Sign test across the 7 models: under H0 "no systematic direction", each
#        model is independently +/−; observing 7/7 positive has p = (1/2)^7.
#    (c) GLMM (logistic): flipped ~ attribution
#         + (attribution|model_short) + (1|original_id) + (1|subject)
#        validates that the effect survives item and subject random effects.
# ═════════════════════════════════════════════════════════════════════════════

section_h("R2  Self-Attribution Delta — paired tests + GLMM (Table 3)")

# Build paired dataset — pair at QUESTION level (matches paper Definition 3.1).
# Each (model, original_id, k) appears once under blind and once under self.
PAIR_KEYS = ["model_short", "original_id", "k"]
sa_blind = self_pool[self_pool["attribution"] == "blind"][
    PAIR_KEYS + ["flipped", "subject"]
].rename(columns={"flipped": "flipped_blind"})
sa_self  = self_pool[self_pool["attribution"] == "self"][
    PAIR_KEYS + ["flipped"]
].rename(columns={"flipped": "flipped_self"})
paired = sa_blind.merge(sa_self, on=PAIR_KEYS, how="inner")
paired["delta"] = paired["flipped_self"] - paired["flipped_blind"]
print(f"\nPaired (blind, self) question-rows: {len(paired):,}")
print(f"Models with both conditions: {sorted(paired['model_short'].unique())}")

# (a) Paired cluster bootstrap per model — clusters = original_id
print("\n[R2a] Paired bootstrap SAD per model (cluster=original_id)")
rows = []
for m in sorted(paired["model_short"].unique()):
    sub = paired[paired["model_short"] == m]
    blind_grouped = sub.groupby("original_id")["flipped_blind"].apply(np.asarray)
    self_grouped  = sub.groupby("original_id")["flipped_self"].apply(np.asarray)
    qids = blind_grouped.index.to_numpy()
    bv = blind_grouped.to_numpy(dtype=object)
    sv = self_grouped.to_numpy(dtype=object)
    n_c = len(qids)
    point = sub["flipped_self"].mean() - sub["flipped_blind"].mean()
    boots = np.empty(N_BOOT)
    for b in range(N_BOOT):
        idx = RNG.integers(0, n_c, size=n_c)
        bb = np.concatenate([bv[i] for i in idx])
        ss = np.concatenate([sv[i] for i in idx])
        boots[b] = ss.mean() - bb.mean()
    lo, hi = ci_pct(boots)
    p_two = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    rows.append({
        "model": m, "n_pairs": len(sub), "n_questions": n_c,
        "AFR_blind": sub["flipped_blind"].mean(),
        "AFR_self":  sub["flipped_self"].mean(),
        "SAD":       point,
        "ci_lo": lo, "ci_hi": hi,
        "p_boot_two_sided": p_two,
    })
sad_ci = pd.DataFrame(rows).sort_values("SAD", ascending=False)
sad_ci["sig"] = sad_ci["p_boot_two_sided"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)
print(sad_ci.assign(
    AFR_blind=lambda d: (d["AFR_blind"] * 100).round(1),
    AFR_self =lambda d: (d["AFR_self"]  * 100).round(1),
    SAD      =lambda d: (d["SAD"]       * 100).round(2),
    ci=lambda d: [fmt_ci(l, h, prec=2) for l, h in zip(d["ci_lo"], d["ci_hi"])],
)[["model", "n_questions", "AFR_blind", "AFR_self", "SAD", "ci",
   "p_boot_two_sided", "sig"]].to_string(index=False))
sad_ci.to_csv(os.path.join(ROB_DIR, "R2a_SAD_paired_bootstrap.csv"), index=False)

# (b) Sign test across models
n_pos = int((sad_ci["SAD"] > 0).sum())
n_total = len(sad_ci)
# Two-sided exact binomial p
from math import comb
def binom_two_sided(k, n, p=0.5):
    # P(X = j) = C(n,j) p^j (1-p)^(n-j)
    pmf = np.array([comb(n, j) * (p ** j) * ((1 - p) ** (n - j)) for j in range(n + 1)])
    obs = pmf[k]
    return float(pmf[pmf <= obs + 1e-12].sum())
p_sign = binom_two_sided(n_pos, n_total)
print(f"\n[R2b] Sign test across {n_total} models: {n_pos}/{n_total} positive  "
      f"two-sided p = {p_sign:.4g}")

# (c) GLMM — logistic mixed model
# Try statsmodels BinomialBayesMixedGLM (variational) → falls back to a
# fixed-effects clustered logit if statsmodels is missing.
print("\n[R2c] Mixed-effects logistic regression")
glmm_long = pd.concat([
    sa_blind.assign(attribution="blind").rename(columns={"flipped_blind": "flipped"}),
    sa_self .assign(attribution="self" ).rename(columns={"flipped_self":  "flipped"}),
], ignore_index=True)
# Need subject for self side too — re-merge subject from blind
subj_lookup = sa_blind[PAIR_KEYS + ["subject"]]
glmm_long = glmm_long.drop(columns=["subject"], errors="ignore").merge(
    subj_lookup, on=PAIR_KEYS, how="left"
)
glmm_long = glmm_long.dropna(subset=["subject"])
glmm_long["attr_self"] = (glmm_long["attribution"] == "self").astype(int)
# Question-level: item == original_id (paper's Definition 3.1 unit).
glmm_long["item_id"]   = glmm_long["original_id"].astype(str)


def _fit_logit_newton(X, y, max_iter=50, tol=1e-7):
    """Vanilla Newton-Raphson logistic regression. Returns beta, hessian-inverse, X."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        eta = X @ beta
        # Numerical guard
        eta = np.clip(eta, -30, 30)
        mu  = 1.0 / (1.0 + np.exp(-eta))
        W   = mu * (1.0 - mu)
        # Hessian = X' diag(W) X; gradient = X'(y - mu)
        XtW = X.T * W
        H = XtW @ X
        g = X.T @ (y - mu)
        # Ridge for numerical stability if needed
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.solve(H + 1e-6 * np.eye(p), g)
        beta_new = beta + step
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break
        beta = beta_new
    eta = np.clip(X @ beta, -30, 30)
    mu  = 1.0 / (1.0 + np.exp(-eta))
    W   = mu * (1.0 - mu)
    H   = (X.T * W) @ X
    H_inv = np.linalg.pinv(H)
    return beta, H_inv, mu


def _cluster_robust_cov(X, y, mu, H_inv, clusters):
    """Liang–Zeger cluster-robust sandwich variance for logit.
    V = H_inv @ (Σ_c (X_c' u_c)(X_c' u_c)') @ H_inv
    """
    X = np.asarray(X, dtype=float)
    u = y - mu  # score residuals
    df = pd.DataFrame(X * u[:, None])
    df["__cl__"] = clusters
    # Sum scores within each cluster, then outer-product and accumulate
    grouped = df.groupby("__cl__").sum().to_numpy()  # (n_clusters, p)
    meat = grouped.T @ grouped
    V = H_inv @ meat @ H_inv
    # Small-cluster correction (Stata-style): n_c / (n_c - 1)
    n_c = grouped.shape[0]
    if n_c > 1:
        V *= n_c / (n_c - 1)
    return V


def _build_design(df, fe_cols):
    """Build [intercept, attr_self, dummies for each fe_col]. Drops one level per FE."""
    parts = [np.ones((len(df), 1)), df[["attr_self"]].to_numpy(dtype=float)]
    names = ["intercept", "attr_self"]
    for col in fe_cols:
        levels = sorted(df[col].unique())[1:]  # drop reference level
        for lv in levels:
            parts.append((df[col] == lv).astype(float).to_numpy()[:, None])
            names.append(f"{col}={lv}")
    return np.hstack(parts), names


from scipy.stats import norm

glmm_summary = None

# Tier 1: Bayesian variational mixed model (best — random intercepts)
try:
    from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
    fml_fix = "flipped ~ attr_self"
    fml_rand = {
        "model":   "0 + C(model_short)",
        "item":    "0 + C(item_id)",
        "subject": "0 + C(subject)",
    }
    print("  [tier 1] Fitting BinomialBayesMixedGLM (variational) — this can take a minute…")
    md = BinomialBayesMixedGLM.from_formula(fml_fix, fml_rand, data=glmm_long)
    fit = md.fit_vb()
    fe_names = fit.model.exog_names
    coef_idx = fe_names.index("attr_self")
    beta = float(fit.fe_mean[coef_idx])
    sd   = float(fit.fe_sd[coef_idx])
    lo, hi = beta - 1.96 * sd, beta + 1.96 * sd
    z = beta / sd
    p = 2 * (1 - norm.cdf(abs(z)))
    glmm_summary = {
        "method": "BinomialBayesMixedGLM (variational; random intercepts: model, item, subject)",
        "fixed_effect_self": beta, "sd": sd,
        "ci_lo": lo, "ci_hi": hi, "z": z, "p": float(p),
        "odds_ratio": float(np.exp(beta)),
        "n": int(len(glmm_long)),
    }
except ModuleNotFoundError:
    pass
except Exception as e:
    print(f"  [tier 1] BinomialBayesMixedGLM failed: {type(e).__name__}: {e}")

# Tier 2: statsmodels Logit with cluster-robust SE on items
if glmm_summary is None:
    try:
        import statsmodels.api as sm
        print("  [tier 2] statsmodels Logit + cluster-robust SE on item_id "
              "(model & subject as fixed effects)")
        # Encode model_short and subject as fixed effects via dummies
        X_df = pd.get_dummies(
            glmm_long[["attr_self", "model_short", "subject"]],
            columns=["model_short", "subject"], drop_first=True
        ).astype(float)
        X = sm.add_constant(X_df)
        y = glmm_long["flipped"].astype(int)
        res = sm.Logit(y, X).fit(disp=False, cov_type="cluster",
                                 cov_kwds={"groups": glmm_long["item_id"].values})
        beta = float(res.params["attr_self"])
        se   = float(res.bse["attr_self"])
        lo, hi = beta - 1.96 * se, beta + 1.96 * se
        glmm_summary = {
            "method": "Logit + cluster-robust SE on item; model & subject as fixed effects",
            "fixed_effect_self": beta, "sd": se,
            "ci_lo": lo, "ci_hi": hi,
            "z": float(beta / se), "p": float(res.pvalues["attr_self"]),
            "odds_ratio": float(np.exp(beta)),
            "n": int(len(glmm_long)),
        }
    except ModuleNotFoundError:
        pass
    except Exception as e:
        print(f"  [tier 2] statsmodels Logit failed: {type(e).__name__}: {e}")

# Tier 3: pure numpy/scipy logit with hand-rolled cluster-robust SE on items
if glmm_summary is None:
    print("  [tier 3] statsmodels not available — using pure numpy/scipy logit "
          "with cluster-robust SE on item_id (Liang–Zeger sandwich)")
    print("           model & subject absorbed as fixed-effect dummies")
    X, names = _build_design(glmm_long, fe_cols=["model_short", "subject"])
    y = glmm_long["flipped"].to_numpy(dtype=float)
    beta_vec, H_inv, mu = _fit_logit_newton(X, y)
    V = _cluster_robust_cov(X, y, mu, H_inv, glmm_long["item_id"].to_numpy())
    se_vec = np.sqrt(np.diag(V))
    j = names.index("attr_self")
    beta = float(beta_vec[j])
    se   = float(se_vec[j])
    lo, hi = beta - 1.96 * se, beta + 1.96 * se
    z = beta / se
    p = 2 * (1 - norm.cdf(abs(z)))
    n_items = glmm_long["item_id"].nunique()
    glmm_summary = {
        "method": (f"numpy/scipy logit + cluster-robust SE on item "
                   f"({n_items:,} clusters); model & subject as fixed effects"),
        "fixed_effect_self": beta, "sd": se,
        "ci_lo": lo, "ci_hi": hi, "z": float(z), "p": float(p),
        "odds_ratio": float(np.exp(beta)),
        "n": int(len(glmm_long)),
    }

print(f"  Method: {glmm_summary['method']}")
print(f"  Fixed effect (self vs blind): β = {glmm_summary['fixed_effect_self']:.4f}  "
      f"SE/SD = {glmm_summary['sd']:.4f}")
print(f"  95% CI: [{glmm_summary['ci_lo']:.4f}, {glmm_summary['ci_hi']:.4f}]  "
      f"z = {glmm_summary['z']:.2f}  p = {glmm_summary['p']:.4g}")
or_lo, or_hi = np.exp(glmm_summary['ci_lo']), np.exp(glmm_summary['ci_hi'])
print(f"  Odds-ratio: {glmm_summary['odds_ratio']:.3f}  "
      f"95% CI: [{or_lo:.3f}, {or_hi:.3f}]")
print(f"  (≈{(glmm_summary['odds_ratio']-1)*100:+.1f}% increase in flip odds "
      "when argument is attributed to self)")


with open(os.path.join(ROB_DIR, "R2c_glmm_self_attribution.json"), "w") as f:
    json.dump(glmm_summary, f, indent=2)


# ═════════════════════════════════════════════════════════════════════════════
#  R3 — Subject-level AFR variation (Table 5 / Figure 3)
#  - Per-subject AFR with Wilson CI (binomial) — cleaner than bootstrap here
#  - Permutation test on the 60-pt spread: is the observed range larger than
#    expected if subject labels were exchangeable across items within a model?
# ═════════════════════════════════════════════════════════════════════════════

section_h("R3  Subject AFR — CIs + permutation test for spread (Table 5)")

# Use the same pool the paper uses for Table 5: avg across models, k, attribution
subj_pool = self_pool.copy()  # already restricted to BC ∩ any-CS, high pressure
print(f"Pool for subject analysis: {len(subj_pool):,} rows  "
      f"(unit = question × model × attribution × k)")

# Wilson CI for proportion
def wilson_ci(k, n, alpha=ALPHA):
    if n == 0:
        return np.nan, np.nan
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    phat = k / n
    denom = 1 + z**2 / n
    centre = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half

subj_rows = []
for s, sub in subj_pool.groupby("subject"):
    n = len(sub); k = int(sub["flipped"].sum())
    lo, hi = wilson_ci(k, n)
    subj_rows.append({"subject": s, "n": n, "AFR": k / n,
                      "ci_lo": lo, "ci_hi": hi})
subj_ci = pd.DataFrame(subj_rows).sort_values("AFR", ascending=False)
print(f"\nObserved AFR range across subjects: "
      f"{subj_ci['AFR'].max()*100:.1f} − {subj_ci['AFR'].min()*100:.1f} = "
      f"{(subj_ci['AFR'].max() - subj_ci['AFR'].min())*100:.1f} pp")

# Print top 10 / bottom 10 with CIs
print("\n  Top 10 most vulnerable (with 95% Wilson CIs):")
print(subj_ci.head(10).assign(
    AFR=lambda d: (d["AFR"]*100).round(1),
    ci=lambda d: [fmt_ci(l, h) for l, h in zip(d["ci_lo"], d["ci_hi"])],
)[["subject", "n", "AFR", "ci"]].to_string(index=False))
print("\n  Top 10 most robust (with 95% Wilson CIs):")
print(subj_ci.tail(10).iloc[::-1].assign(
    AFR=lambda d: (d["AFR"]*100).round(1),
    ci=lambda d: [fmt_ci(l, h) for l, h in zip(d["ci_lo"], d["ci_hi"])],
)[["subject", "n", "AFR", "ci"]].to_string(index=False))
subj_ci.to_csv(os.path.join(ROB_DIR, "R3_subject_AFR_with_CI.csv"), index=False)

# Permutation test on observed range
# H0: subject labels are exchangeable WITHIN (model, k, attribution) blocks
# (this respects model-level baseline differences). We shuffle the subject
# labels within each block and recompute the max−min across subjects.
print("\n[R3b] Permutation test on observed subject-AFR range")
print(f"      H0: subjects exchangeable within (model, k, attribution) blocks")
N_PERM = min(N_BOOT, 1000)
obs_range = subj_ci["AFR"].max() - subj_ci["AFR"].min()

block_keys = ["model_short", "k", "attribution"]
perm_ranges = np.empty(N_PERM)
# Reset index so block positions match positional indexing into the numpy arrays
subj_pool_arr = (
    subj_pool[["subject", "flipped"] + block_keys]
    .reset_index(drop=True)
    .copy()
)
subj_arr     = subj_pool_arr["subject"].to_numpy()
flipped_arr  = subj_pool_arr["flipped"].to_numpy()
# Pre-cache positional indices per block
block_idx = {k: g.index.to_numpy() for k, g in subj_pool_arr.groupby(block_keys)}

for r in range(N_PERM):
    permuted = subj_arr.copy()
    for _, idx in block_idx.items():
        if len(idx) > 1:
            permuted[idx] = RNG.permutation(subj_arr[idx])
    tmp = pd.Series(flipped_arr).groupby(permuted).mean()
    perm_ranges[r] = tmp.max() - tmp.min()
p_perm = (perm_ranges >= obs_range).mean()
print(f"      Observed range = {obs_range*100:.1f} pp")
print(f"      Null range μ = {perm_ranges.mean()*100:.1f} pp, "
      f"95th pct = {np.percentile(perm_ranges, 95)*100:.1f} pp")
print(f"      p (one-sided, observed ≥ null) = {p_perm:.4g}")


# ═════════════════════════════════════════════════════════════════════════════
#  R4 — Variance decomposition CIs (Section 5.6, "77.7% / 13.3% / 8.1%")
#  Bootstrap over (cm_short, bm_short, subject) cells, recompute SS each draw.
#  This is the same unit the original code uses to compute the point estimates.
# ═════════════════════════════════════════════════════════════════════════════

section_h("R4  Variance decomposition CIs — cross-model 77.7/13.3/8.1 claim")

# Reproduce the cell-level data the original script builds (k=10 question-level)
cross_pool = flat_cross[flat_cross["coercion_k"] == 10].copy()
question_lvl = (
    cross_pool
    .groupby(["cm_short", "bm_short", "original_id", "subject"])
    .agg(flipped=("collapsed", "any"))
    .reset_index()
)
# Aggregate to (cm, bm, subject) cells
cells = (
    question_lvl
    .groupby(["cm_short", "bm_short", "subject"])
    .agg(cmfr=("flipped", "mean"), n=("flipped", "count"))
    .reset_index()
)
print(f"Cells (cm × bm × subject): {len(cells):,}")

def variance_decomp(df):
    """Return (frac_baseline, frac_source, frac_subject, frac_resid)."""
    if len(df) == 0:
        return (np.nan,) * 4
    grand = df["cmfr"].mean()
    ss_total = ((df["cmfr"] - grand) ** 2).sum()
    if ss_total == 0:
        return (np.nan,) * 4
    def ss_factor(col):
        means = df.groupby(col)["cmfr"].mean()
        sizes = df.groupby(col).size()
        return float(((means - grand) ** 2 * sizes).sum())
    ss_subj = ss_factor("subject")
    ss_src  = ss_factor("cm_short")
    ss_bsl  = ss_factor("bm_short")
    ss_res  = max(ss_total - ss_subj - ss_src - ss_bsl, 0.0)
    return (ss_bsl / ss_total, ss_src / ss_total,
            ss_subj / ss_total, ss_res / ss_total)

vd_point = variance_decomp(cells)
print(f"\nPoint estimates (matches the 77.7 / 13.3 / 8.1 / residual numbers):")
print(f"  Baseline: {vd_point[0]*100:5.1f}%")
print(f"  Source:   {vd_point[1]*100:5.1f}%")
print(f"  Subject:  {vd_point[2]*100:5.1f}%")
print(f"  Residual: {vd_point[3]*100:5.1f}%")

# Bootstrap: resample CELLS with replacement
print(f"\n[R4] Bootstrapping {N_BOOT} replicates over cells…")
vd_boot = np.empty((N_BOOT, 4))
n_cells = len(cells)
for b in range(N_BOOT):
    samp = cells.iloc[RNG.integers(0, n_cells, size=n_cells)]
    vd_boot[b] = variance_decomp(samp)

labels = ["Baseline", "Source", "Subject", "Residual"]
print(f"\n{'Component':<10}  {'Point':>7}  {'95% CI':>20}  {'Boot SD':>9}")
print("─" * 52)
vd_rows = []
for i, lab in enumerate(labels):
    lo, hi = ci_pct(vd_boot[:, i])
    sd = vd_boot[:, i].std()
    print(f"{lab:<10}  {vd_point[i]*100:6.1f}%  [{lo*100:5.1f}, {hi*100:5.1f}]   {sd*100:7.2f}pp")
    vd_rows.append({"component": lab, "point": vd_point[i],
                    "ci_lo": lo, "ci_hi": hi, "sd": sd})
pd.DataFrame(vd_rows).to_csv(
    os.path.join(ROB_DIR, "R4_variance_decomposition_CI.csv"), index=False
)

# Check: does the 95% CI for Baseline strictly exceed Source's 95% CI?
overlap = ci_pct(vd_boot[:, 0])[0] > ci_pct(vd_boot[:, 1])[1]
print(f"\nNon-overlap (Baseline_lo > Source_hi): {overlap}")
print("  → the dominance of baseline susceptibility is not a sampling artifact"
      if overlap else
      "  → CIs overlap; report this honestly.")


# ═════════════════════════════════════════════════════════════════════════════
#  R5 — Cross vs Self ∆ per model (Table 6) with paired-item CIs + FDR
# ═════════════════════════════════════════════════════════════════════════════

section_h("R5  Cross vs Self ∆ per model — paired item bootstrap (Table 6)")

# For each baseline model B, compare AFR(blind, k=10, self-source on B) vs.
# AFR over all cross sources challenging B at k=10. Pair on original_id (question-level,
# matching paper Definition 3.1). Cross is aggregated to question level by `any` across
# the 3 wrong options (consistent with self side's definition).
self_k10 = self_pool[(self_pool["attribution"] == "blind") & (self_pool["k"] == 10)][
    ["model_short", "original_id", "flipped"]
].rename(columns={"model_short": "bm_short", "flipped": "flipped_self"})

cross_k10 = flat_cross[flat_cross["coercion_k"] == 10].copy()
# Question-level cross flip per (cm, bm, item): any wrong option flips the question
cross_q = (
    cross_k10[cross_k10["cm_short"] != cross_k10["bm_short"]]
    .groupby(["cm_short", "bm_short", "original_id"])
    .agg(flipped_cross=("collapsed", "any"))
    .reset_index()
)
cross_q["flipped_cross"] = cross_q["flipped_cross"].astype(int)
# Average across cross-source models for each (bm, item)
cross_avg = (
    cross_q.groupby(["bm_short", "original_id"])["flipped_cross"]
    .mean().reset_index()
)
joined = self_k10.merge(cross_avg, on=["bm_short", "original_id"], how="inner")
print(f"\nPaired (self k=10 blind, cross-avg) question-rows: {len(joined):,}")

cross_rows = []
p_values = []
for m in sorted(joined["bm_short"].unique()):
    sub = joined[joined["bm_short"] == m]
    grp_self  = sub.groupby("original_id")["flipped_self"].apply(np.asarray)
    grp_cross = sub.groupby("original_id")["flipped_cross"].apply(np.asarray)
    qids = grp_self.index.to_numpy()
    sv = grp_self.to_numpy(dtype=object)
    cv = grp_cross.to_numpy(dtype=object)
    n_c = len(qids)
    point = sub["flipped_cross"].mean() - sub["flipped_self"].mean()
    boots = np.empty(N_BOOT)
    for b in range(N_BOOT):
        idx = RNG.integers(0, n_c, size=n_c)
        ss = np.concatenate([sv[i] for i in idx])
        cc = np.concatenate([cv[i] for i in idx])
        boots[b] = cc.mean() - ss.mean()
    lo, hi = ci_pct(boots)
    p_two = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    cross_rows.append({"model": m, "n_questions": n_c,
                       "AFR_self": sub["flipped_self"].mean(),
                       "AFR_cross_avg": sub["flipped_cross"].mean(),
                       "delta": point, "ci_lo": lo, "ci_hi": hi,
                       "p_boot": p_two})
    p_values.append(p_two)

# Benjamini-Hochberg FDR correction
def bh_fdr(pvals, alpha=0.05):
    p = np.asarray(pvals)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * n / (np.arange(n) + 1)
    # Make monotone non-decreasing
    adj_mono = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(adj_mono)
    out[order] = adj_mono
    return np.clip(out, 0, 1)

cross_ci = pd.DataFrame(cross_rows)
cross_ci["q_BH"] = bh_fdr(cross_ci["p_boot"].to_numpy())
cross_ci["sig_q"] = cross_ci["q_BH"].apply(
    lambda q: "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"
)
cross_ci = cross_ci.sort_values("delta", ascending=False)
print(cross_ci.assign(
    AFR_self=lambda d: (d["AFR_self"] * 100).round(1),
    AFR_cross_avg=lambda d: (d["AFR_cross_avg"] * 100).round(1),
    delta=lambda d: (d["delta"] * 100).round(2),
    ci=lambda d: [fmt_ci(l, h, prec=2) for l, h in zip(d["ci_lo"], d["ci_hi"])],
    p_boot=lambda d: d["p_boot"].round(4),
    q_BH=lambda d: d["q_BH"].round(4),
)[["model", "n_questions", "AFR_self", "AFR_cross_avg", "delta", "ci",
   "p_boot", "q_BH", "sig_q"]].to_string(index=False))
cross_ci.to_csv(os.path.join(ROB_DIR, "R5_cross_vs_self_paired.csv"), index=False)


# ═════════════════════════════════════════════════════════════════════════════
#  R6 — MAXFLIP gain over standard self-source (Table 7) with paired CIs
# ═════════════════════════════════════════════════════════════════════════════

section_h("R6  MAXFLIP ∆ — paired item bootstrap (Table 7)")

# MAXFLIP construction (mirrors paper §5.7): per question, choose the cross-source
# argument that flips the most baseline models; ties broken randomly.
# Then per model, compare AFR(self k=10 blind) vs AFR(maxflip).
print("\nReconstructing MAXFLIP from cross-model pool…")
# Per (cm, original_id) compute how many bm's flipped (any wrong option suffices)
ck10 = flat_cross[flat_cross["coercion_k"] == 10].copy()
# Question-level flip per (cm, bm, item)
qflip = (
    ck10
    .groupby(["cm_short", "bm_short", "original_id"])
    .agg(any_flip=("collapsed", "any"))
    .reset_index()
)
# How many bm's a given (cm, item) flips
score = (
    qflip[qflip["cm_short"] != qflip["bm_short"]]
    .groupby(["cm_short", "original_id"])
    .agg(n_flipped=("any_flip", "sum"), n_targets=("any_flip", "count"))
    .reset_index()
)
# For each item, choose argmax cm; random tiebreak
score["tiebreak"] = RNG.random(len(score))
score = score.sort_values(["original_id", "n_flipped", "tiebreak"],
                          ascending=[True, False, True])
maxflip_choice = score.drop_duplicates("original_id", keep="first")[
    ["original_id", "cm_short"]
].rename(columns={"cm_short": "maxflip_cm"})
print(f"  MAXFLIP arguments selected for {len(maxflip_choice):,} questions")
print("  Producer share:")
print(maxflip_choice["maxflip_cm"].value_counts(normalize=True).mul(100).round(1).to_string())

# For each baseline bm, get item-level flips when challenged by MAXFLIP
mf_rows_all = []
for bm in sorted(ck10["bm_short"].unique()):
    bm_data = ck10[ck10["bm_short"] == bm].merge(
        maxflip_choice, on="original_id", how="inner"
    )
    bm_data = bm_data[bm_data["cm_short"] == bm_data["maxflip_cm"]]
    # Question-level: any wrong-option flip counts (matching paper convention)
    qmf = (
        bm_data
        .groupby("original_id")
        .agg(flipped_max=("collapsed", "any"))
        .reset_index()
    )
    qmf["flipped_max"] = qmf["flipped_max"].astype(int)
    qmf["bm_short"] = bm
    mf_rows_all.append(qmf)
maxflip_q = pd.concat(mf_rows_all, ignore_index=True)

# Self k=10 blind, question-level
self_q = (
    self_pool[(self_pool["attribution"] == "blind") & (self_pool["k"] == 10)]
    .groupby(["model_short", "original_id"])
    .agg(flipped_self=("flipped", "max"))
    .reset_index()
    .rename(columns={"model_short": "bm_short"})
)
mf_join = self_q.merge(maxflip_q, on=["bm_short", "original_id"], how="inner")
print(f"\nPaired (self_blind_k10, MAXFLIP) question-rows: {len(mf_join):,}")

mf_rows = []
for m in sorted(mf_join["bm_short"].unique()):
    sub = mf_join[mf_join["bm_short"] == m]
    point = sub["flipped_max"].mean() - sub["flipped_self"].mean()
    n = len(sub)
    boots = np.empty(N_BOOT)
    s = sub["flipped_self"].to_numpy(); c = sub["flipped_max"].to_numpy()
    for b in range(N_BOOT):
        idx = RNG.integers(0, n, size=n)
        boots[b] = c[idx].mean() - s[idx].mean()
    lo, hi = ci_pct(boots)
    p_two = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    mf_rows.append({"model": m, "n_questions": n,
                    "AFR_self_k10": sub["flipped_self"].mean(),
                    "AFR_maxflip":  sub["flipped_max"].mean(),
                    "gain": point, "ci_lo": lo, "ci_hi": hi, "p_boot": p_two})
mf_ci = pd.DataFrame(mf_rows).sort_values("gain", ascending=False)
mf_ci["q_BH"] = bh_fdr(mf_ci["p_boot"].to_numpy())
print(mf_ci.assign(
    AFR_self_k10=lambda d: (d["AFR_self_k10"] * 100).round(1),
    AFR_maxflip =lambda d: (d["AFR_maxflip"]  * 100).round(1),
    gain=lambda d: (d["gain"] * 100).round(2),
    ci=lambda d: [fmt_ci(l, h, prec=2) for l, h in zip(d["ci_lo"], d["ci_hi"])],
)[["model", "n_questions", "AFR_self_k10", "AFR_maxflip", "gain", "ci",
   "p_boot", "q_BH"]].to_string(index=False))
mf_ci.to_csv(os.path.join(ROB_DIR, "R6_maxflip_gain_paired.csv"), index=False)


# ═════════════════════════════════════════════════════════════════════════════
#  CONSOLIDATED FIGURE: forest plots for the headline claims
# ═════════════════════════════════════════════════════════════════════════════

section_h("Forest-plot figure for the appendix")

fig, axes = plt.subplots(1, 4, figsize=(18, 5.5))
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
})

# Panel 1: AFR per model
ax = axes[0]
df = afr_blind_ci.copy()
y = np.arange(len(df))[::-1]
ax.errorbar(df["AFR"]*100, y,
            xerr=[(df["AFR"] - df["ci_lo"])*100, (df["ci_hi"] - df["AFR"])*100],
            fmt="o", color="#2D2D2D", ecolor="#7F7F7F",
            elinewidth=1.2, capsize=3, markersize=6)
ax.set_yticks(y); ax.set_yticklabels(df["model"], fontsize=9)
ax.set_xlabel("AFR_blind (%)  with 95% cluster-bootstrap CI")
ax.set_title("(a) Per-model AFR\nclusters = original_id", fontweight="bold")
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 105)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

# Panel 2: SAD per model with zero line
ax = axes[1]
df = sad_ci.copy()
y = np.arange(len(df))[::-1]
colors = ["#27ae60" if s > 0 and p < 0.05 else "#888888"
          for s, p in zip(df["SAD"], df["p_boot_two_sided"])]
ax.errorbar(df["SAD"]*100, y,
            xerr=[(df["SAD"] - df["ci_lo"])*100, (df["ci_hi"] - df["SAD"])*100],
            fmt="o", ecolor="#7F7F7F", elinewidth=1.2, capsize=3, markersize=6,
            mfc="none", mec="black")
for i, c in enumerate(colors):
    ax.plot(df["SAD"].iloc[i]*100, y[i], "o", color=c, markersize=7, zorder=5)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.7)
ax.set_yticks(y); ax.set_yticklabels(df["model"], fontsize=9)
ax.set_xlabel("SAD = AFR_self − AFR_blind (pp)")
ax.set_title("(b) Self-Attribution Delta\npaired bootstrap, sign p = "
             f"{p_sign:.3g} (7/7 positive)", fontweight="bold")
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

# Panel 3: Variance decomposition CIs
ax = axes[2]
labels_vd = ["Baseline\n(challenged)", "Source\n(author)", "Subject", "Residual"]
y = np.arange(len(labels_vd))[::-1]
pts = [vd_point[i]*100 for i in range(4)]
errs_lo = [(vd_point[i] - ci_pct(vd_boot[:, i])[0])*100 for i in range(4)]
errs_hi = [(ci_pct(vd_boot[:, i])[1] - vd_point[i])*100 for i in range(4)]
colors_vd = ["#c0392b", "#e67e22", "#3498db", "#95a5a6"]
ax.errorbar(pts, y, xerr=[errs_lo, errs_hi], fmt="o",
            color="#2D2D2D", ecolor="#7F7F7F", elinewidth=1.5,
            capsize=4, markersize=7, zorder=4)
for i, c in enumerate(colors_vd):
    ax.plot(pts[i], y[i], "o", color=c, markersize=9, zorder=5,
            mec="black", mew=0.8)
ax.set_yticks(y); ax.set_yticklabels(labels_vd, fontsize=9)
ax.set_xlabel("% of variance in cell-level CMFR")
ax.set_title("(c) Variance decomposition\n(cm × bm × subject) cells, k=10 blind",
             fontweight="bold")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

# Panel 4: Cross-self delta per model
ax = axes[3]
df = cross_ci.copy()
y = np.arange(len(df))[::-1]
colors_cs = ["#27ae60" if d > 0 and q < 0.05 else
             "#c0392b" if d < 0 and q < 0.05 else "#888888"
             for d, q in zip(df["delta"], df["q_BH"])]
ax.errorbar(df["delta"]*100, y,
            xerr=[(df["delta"] - df["ci_lo"])*100, (df["ci_hi"] - df["delta"])*100],
            fmt="o", ecolor="#7F7F7F", elinewidth=1.2, capsize=3, markersize=6,
            mfc="none", mec="black")
for i, c in enumerate(colors_cs):
    ax.plot(df["delta"].iloc[i]*100, y[i], "o", color=c, markersize=7, zorder=5)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.7)
ax.set_yticks(y); ax.set_yticklabels(df["model"], fontsize=9)
ax.set_xlabel("∆ = AFR_cross − AFR_self (pp)")
ax.set_title("(d) Cross vs Self per model\npaired, BH-FDR corrected",
             fontweight="bold")
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig_path = os.path.join(OUT_DIR, "robustness_forest.pdf")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [fig] {fig_path}")


# ═════════════════════════════════════════════════════════════════════════════
#  LaTeX snippets — drop-in for the appendix
# ═════════════════════════════════════════════════════════════════════════════

section_h("LaTeX snippets")

def tex_afr_blind():
    lines = [
        r"\begin{table}[t]\centering\small",
        r"\caption{Per-model AFR$_{\text{blind}}$ with 95\% cluster bootstrap CIs "
        r"(clusters = MMLU question, $B{=}" + str(N_BOOT) + r"$).}",
        r"\label{tab:afr-ci}",
        r"\begin{tabular}{lrrr}\toprule",
        r"Model & $n_q$ & AFR\,(\%) & 95\% CI \\\midrule",
    ]
    for _, r in afr_blind_ci.iterrows():
        lines.append(f"{r['model']} & {int(r['n_questions'])} & "
                     f"{r['AFR']*100:.1f} & "
                     f"[{r['ci_lo']*100:.1f}, {r['ci_hi']*100:.1f}] \\\\")
    lines += [r"\bottomrule\end{tabular}\end{table}"]
    return "\n".join(lines)

def tex_sad():
    lines = [
        r"\begin{table}[t]\centering\small",
        r"\caption{Self-Attribution Delta with paired cluster-bootstrap 95\% CIs "
        r"and two-sided bootstrap $p$-values. Sign test across models: "
        f"{n_pos}/{n_total}" + r" positive, $p_{\text{sign}}=" +
        f"{p_sign:.3g}" + r"$. GLMM odds-ratio for self attribution: "
        f"{glmm_summary['odds_ratio']:.2f} "
        f"[CI {np.exp(glmm_summary['ci_lo']):.2f}, "
        f"{np.exp(glmm_summary['ci_hi']):.2f}]" + r".}",
        r"\label{tab:sad-ci}",
        r"\begin{tabular}{lrrrrl}\toprule",
        r"Model & $n_q$ & AFR$_{\text{blind}}$ & AFR$_{\text{self}}$ & SAD\,(pp) & 95\% CI \\\midrule",
    ]
    for _, r in sad_ci.iterrows():
        lines.append(
            f"{r['model']} & {int(r['n_questions'])} & "
            f"{r['AFR_blind']*100:.1f} & {r['AFR_self']*100:.1f} & "
            f"{r['SAD']*100:+.1f}{r['sig']} & "
            f"[{r['ci_lo']*100:+.1f}, {r['ci_hi']*100:+.1f}] \\\\"
        )
    lines += [r"\bottomrule\end{tabular}\end{table}"]
    return "\n".join(lines)

def tex_vd():
    rows = [(labels[i], vd_point[i], *ci_pct(vd_boot[:, i])) for i in range(4)]
    lines = [
        r"\begin{table}[t]\centering\small",
        r"\caption{Variance decomposition of cell-level CMFR over (cm, bm, subject), "
        r"$k{=}10$, with 95\% bootstrap CIs over " + str(N_BOOT) + r" replicates "
        r"(resampling cells).}",
        r"\label{tab:vd-ci}",
        r"\begin{tabular}{lrr}\toprule",
        r"Component & Estimate\,(\%) & 95\% CI \\\midrule",
    ]
    for lab, p, lo, hi in rows:
        lines.append(f"{lab} & {p*100:.1f} & [{lo*100:.1f}, {hi*100:.1f}] \\\\")
    lines += [r"\bottomrule\end{tabular}\end{table}"]
    return "\n".join(lines)

print("\n" + "─" * 72)
print("  LaTeX — R1: per-model AFR with cluster-bootstrap CIs")
print("─" * 72)
print(tex_afr_blind())

print("\n" + "─" * 72)
print("  LaTeX — R2: Self-Attribution Delta (paired bootstrap + GLMM odds)")
print("─" * 72)
print(tex_sad())

print("\n" + "─" * 72)
print("  LaTeX — R4: Variance decomposition with bootstrap CIs")
print("─" * 72)
print(tex_vd())

print(f"\nCSV outputs → {ROB_DIR}/")
print("Done.")

In [ ]:
cross_q.groupby('bm_short').mean()

In [ ]:
section_h("R5  Cross vs Self ∆ per model — paired item bootstrap (Table 6) [FIXED]")

# Correct approach:
# - AFR_blind: k=10, blind, question-level (from self_pool)
# - AFR_cross: k=10, cross, question-level averaged over source models (from flat_cross)
# - Each side uses its OWN full question set (no inner join for point estimates)
# - Bootstrap pairs on original_id within each side separately, then takes difference

# Self side: k=10, blind, question-level
self_k10 = (
    self_pool[(self_pool["attribution"] == "blind") & (self_pool["k"] == 10)]
    .groupby(["model_short", "original_id"])
    .agg(flipped_self=("flipped", "max"))
    .reset_index()
    .rename(columns={"model_short": "bm_short"})
)

# Cross side: k=10, question-level, averaged over source models per (bm, item)
cross_k10 = flat_cross[flat_cross["coercion_k"] == 10].copy()
cross_q = (
    cross_k10[cross_k10["cm_short"] != cross_k10["bm_short"]]
    .groupby(["cm_short", "bm_short", "original_id"])
    .agg(flipped_cross=("collapsed", "any"))
    .reset_index()
)
cross_q["flipped_cross"] = cross_q["flipped_cross"].astype(int)
# Average over source models per (bm, item)
cross_avg = (
    cross_q.groupby(["bm_short", "original_id"])["flipped_cross"]
    .mean().reset_index()
)

# Verify point estimates match unpaired groupby (sanity check)
print("\nSanity check — cross means should match cross_q.groupby('bm_short').mean():")
print(cross_avg.groupby("bm_short")["flipped_cross"].mean().mul(100).round(1))

cross_rows = []
for m in sorted(self_k10["bm_short"].unique()):
    # Self side for this model
    s = self_k10[self_k10["bm_short"] == m].set_index("original_id")["flipped_self"]
    # Cross side for this model
    c = cross_avg[cross_avg["bm_short"] == m].set_index("original_id")["flipped_cross"]

    afr_self  = s.mean()
    afr_cross = c.mean()
    point     = afr_cross - afr_self

    # Bootstrap each side independently over its own question set
    s_arr = s.to_numpy()
    c_arr = c.to_numpy()
    n_s   = len(s_arr)
    n_c   = len(c_arr)

    boots = np.empty(N_BOOT)
    for b in range(N_BOOT):
        boots[b] = (c_arr[RNG.integers(0, n_c, size=n_c)].mean()
                  - s_arr[RNG.integers(0, n_s, size=n_s)].mean())

    lo, hi = ci_pct(boots)
    p_two  = 2 * min((boots <= 0).mean(), (boots >= 0).mean())

    cross_rows.append({
        "model":         m,
        "n_self":        n_s,
        "n_cross":       n_c,
        "AFR_self":      afr_self,
        "AFR_cross_avg": afr_cross,
        "delta":         point,
        "ci_lo":         lo,
        "ci_hi":         hi,
        "p_boot":        p_two,
    })

cross_ci = pd.DataFrame(cross_rows).sort_values("delta", ascending=False)
cross_ci["sig"] = cross_ci["p_boot"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)
print(cross_ci.assign(
    AFR_self      = lambda d: (d["AFR_self"]      * 100).round(1),
    AFR_cross_avg = lambda d: (d["AFR_cross_avg"] * 100).round(1),
    delta         = lambda d: (d["delta"]         * 100).round(2),
    ci            = lambda d: [fmt_ci(l, h, prec=2) for l, h in zip(d["ci_lo"], d["ci_hi"])],
    p_boot        = lambda d: d["p_boot"].round(4),
)[["model", "n_self", "n_cross", "AFR_self", "AFR_cross_avg", "delta", "ci",
   "p_boot", "sig"]].to_string(index=False))
cross_ci.to_csv(os.path.join(ROB_DIR, "R5_cross_vs_self_paired_FIXED.csv"), index=False)

In [ ]:
section_h("R6  MAXFLIP ∆ — independent bootstrap (Table 7) [FIXED]")

# MAXFLIP flip rates come from best_coercion.csv (pre-built)
# Self side: k=10, blind, question-level (same as R5)
# Each side resampled independently — no inner join for point estimates

df_mf = pd.read_csv("best_coercion.csv")

meta_cols_mf = {
    "original_id","subject","question","choices",
    "correct_label","correct_text",
    "coercion_model","coercion_text","score"
}
model_cols_mf = [c for c in df_mf.columns if c not in meta_cols_mf]

short_names = {
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "meta-llama/Llama-3.3-70B-Instruct": "Llama-3.3-70B",
    "Qwen/Qwen3.5-4B":                   "Qwen3.5-4B",
    "Qwen/Qwen3.5-9B":                   "Qwen3.5-9B",
    "google/gemma-4-26B-A4B-it":         "Gemma-4-26B",
    "Qwen/Qwen3.5-35B-A3B":              "Qwen3.5-35B",
    "gpt-5.1":                           "GPT-5.1",
}

# Build MAXFLIP question-level flipped arrays per model
mf_by_model = {}
for col in model_cols_mf:
    m = short_names.get(col, col)
    flipped_arr = (df_mf[col] == "flipped").astype(int).to_numpy()
    eligible    = (df_mf[col].isin(["flipped","held"])).to_numpy()
    mf_by_model[m] = flipped_arr[eligible]   # only eligible questions

mf_rows = []
for m in sorted(self_k10["bm_short"].unique()):
    if m not in mf_by_model:
        print(f"  WARNING: {m} not found in best_coercion.csv — skipping")
        continue

    # Self side
    s_arr = self_k10[self_k10["bm_short"] == m]["flipped_self"].to_numpy()
    # MAXFLIP side
    c_arr = mf_by_model[m]

    afr_self  = s_arr.mean()
    afr_maxflip = c_arr.mean()
    point     = afr_maxflip - afr_self

    n_s = len(s_arr)
    n_c = len(c_arr)

    boots = np.empty(N_BOOT)
    for b in range(N_BOOT):
        boots[b] = (c_arr[RNG.integers(0, n_c, size=n_c)].mean()
                  - s_arr[RNG.integers(0, n_s, size=n_s)].mean())

    lo, hi = ci_pct(boots)
    p_two  = 2 * min((boots <= 0).mean(), (boots >= 0).mean())

    mf_rows.append({
        "model":       m,
        "n_self":      n_s,
        "n_maxflip":   n_c,
        "AFR_self":    afr_self,
        "AFR_maxflip": afr_maxflip,
        "gain":        point,
        "ci_lo":       lo,
        "ci_hi":       hi,
        "p_boot":      p_two,
    })

mf_ci = pd.DataFrame(mf_rows).sort_values("gain", ascending=False)
mf_ci["sig"] = mf_ci["p_boot"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)

print(mf_ci.assign(
    AFR_self    = lambda d: (d["AFR_self"]    * 100).round(1),
    AFR_maxflip = lambda d: (d["AFR_maxflip"] * 100).round(1),
    gain        = lambda d: (d["gain"]        * 100).round(2),
    ci          = lambda d: [fmt_ci(l, h, prec=2) for l, h in zip(d["ci_lo"], d["ci_hi"])],
    p_boot      = lambda d: d["p_boot"].round(4),
)[["model","n_self","n_maxflip","AFR_self","AFR_maxflip","gain","ci","p_boot","sig"]]
  .to_string(index=False))

mf_ci.to_csv(os.path.join(ROB_DIR, "R6_maxflip_gain_FIXED.csv"), index=False)